In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, "|", len(files), "files |", dirs[:5])
    if root.count('/') > 5:
        break

In [ ]:
import os, shutil

SRC = '/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data'
DST = '/kaggle/working/dl/data'
os.makedirs(DST, exist_ok=True)

for f in os.listdir(SRC):
    if f.endswith('.jsonl'):
        shutil.copy(f"{SRC}/{f}", f"{DST}/{f}")


if not os.path.exists(f"{DST}/img"):
    os.symlink(f"{SRC}/img", f"{DST}/img")
!wget -q -O {DST}/test.jsonl https://huggingface.co/datasets/neuralcatcher/hateful_memes/resolve/main/test_seen.jsonl

print("files:", os.listdir(DST))

import json
for split in ['train', 'dev', 'test']:
    rows = [json.loads(l) for l in open(f"{DST}/{split}.jsonl")]
    missing = [x for x in rows if not os.path.exists(f"{DST}/{x['img']}")]
    print(f"{split}: {len(rows)} rows | label: {'label' in rows[0]} | missing images: {len(missing)}")

In [ ]:
import os
REPO = '/kaggle/working/Deep_Correlated_Prompting'
if not os.path.exists(REPO):
    !git clone -q https://github.com/hulianyuyy/Deep_Correlated_Prompting.git {REPO}

%cd {REPO}
!sed -i 's/from pytorch_lightning.metrics import Metric/from torchmetrics import Metric/' clip/gadgets/my_metrics.py
!sed -i '/prepare_data_per_node=False,/d' run.py
!sed -i 's/deterministic=True/deterministic=False/' run.py
!sed -i 's/if pl_module.trainer.max_steps is None:/if pl_module.trainer.max_steps is None or pl_module.trainer.max_steps == -1:/' clip/modules/clip_utils.py

print("PATCH CHECK:")
!grep -n "import Metric" clip/gadgets/my_metrics.py
!grep -n "deterministic" run.py
!grep -n "max_steps is None" clip/modules/clip_utils.py
!grep -c "prepare_data_per_node" run.py

!pip install -q jsonlines
!python make_arrow.py --dataset hateful_memes --root /kaggle/working/dl --target /kaggle/working/dl/arrow
print("\nARROW FILES:")
!ls -lh /kaggle/working/dl/arrow

In [ ]:
!pip install -q uv
!uv venv --python 3.9 /kaggle/working/dcpenv

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

print("\nTORCH CHECK:")
!/kaggle/working/dcpenv/bin/python -c "import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())"

In [ ]:
!uv pip install --python /kaggle/working/dcpenv/bin/python \
  "numpy==1.23.5" "setuptools==65.6.3" \
  pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 \
  timm==0.6.13 sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 \
  "Pillow==9.5.0" einops ftfy jsonlines regex tqdm

print("\nFULL CHECK:")
!/kaggle/working/dcpenv/bin/python -c "import torch, numpy, pytorch_lightning as pl, timm, transformers; print('torch', torch.__version__, '| cuda', torch.cuda.is_available()); print('numpy', numpy.__version__, '| pl', pl.__version__, '| timm', timm.__version__, '| transformers', transformers.__version__)"

In [ ]:
%cd /kaggle/working/Deep_Correlated_Prompting
!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes exp_name=smoke fast_dev_run=True \
    log_dir=/kaggle/working/dcp_results

In [ ]:
import time
%cd /kaggle/working/Deep_Correlated_Prompting
start = time.time()
!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes exp_name=base_fixed \
    max_epoch=20 val_check_interval=1.0 \
    log_dir=/kaggle/working/dcp_results
print(f"\n=== TOTAL: {(time.time()-start)/60:.1f} MINUTES ===")

In [ ]:
import glob, os
%cd /kaggle/working/Deep_Correlated_Prompting
os.makedirs('records/hatememes', exist_ok=True)

ckpts = [c for c in glob.glob('/kaggle/working/dcp_results/base_fixed_seed0/version_*/checkpoints/*.ckpt')
         if 'last' not in os.path.basename(c)]
BEST = sorted(ckpts)[-1]
print("BEST CHECKPOINT:", BEST)

for t in ['image', 'text', 'both']:
    print(f"\n{'='*25} TEST: missing {t} {'='*25}")
    !/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={BEST} test_only=True \
        test_ratio=0.7 test_type={t} test_exp_name=base exp_name=eval_{t} \
        log_dir=/kaggle/working/dcp_results

print("\n=== BASELINE RESULTS ===")
!cat records/hatememes/*

In [ ]:
import glob
for f in sorted(glob.glob('/kaggle/working/Deep_Correlated_Prompting/records/hatememes/*')):
    print(f.split('/')[-1])
    print("   ", open(f).read().strip())

In [ ]:
import glob, os
%cd /kaggle/working/Deep_Correlated_Prompting
ckpts = [c for c in glob.glob('/kaggle/working/dcp_results/base_fixed_seed0/version_*/checkpoints/*.ckpt')
         if 'last' not in os.path.basename(c)]
BEST = sorted(ckpts)[-1]

!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes load_path={BEST} test_only=True \
    test_ratio=0.0 test_type=both exp_name=eval_complete \
    log_dir=/kaggle/working/dcp_results

In [ ]:
import torch, glob
for f in sorted(glob.glob('/kaggle/working/Deep_Correlated_Prompting/datasets/missing_tables/*.pt')):
    t = torch.load(f)
    print(f.split('/')[-1])
    print(f"    total={len(t)}  complete(0)={int((t==0).sum())}  text_missing(1)={int((t==1).sum())}  image_missing(2)={int((t==2).sum())}")

In [ ]:
import glob, os
!chmod -R +x /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting

ckpts = [c for c in glob.glob('/kaggle/working/dcp_results/base_fixed_seed0/version_*/checkpoints/*.ckpt')
         if 'last' not in os.path.basename(c)]
BEST = sorted(ckpts)[-1]
print("BEST:", BEST)

!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes load_path={BEST} test_only=True \
    test_ratio=0.0 test_type=both exp_name=eval_complete \
    log_dir=/kaggle/working/dcp_results

In [ ]:
import glob, os
!chmod -R +x /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting

ckpts = [c for c in glob.glob('/kaggle/working/dcp_results/base_fixed_seed0/version_*/checkpoints/*.ckpt')
         if 'last' not in os.path.basename(c)]
BEST = sorted(ckpts)[-1]

!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes load_path={BEST} test_only=True \
    test_ratio=0.0 test_type=both test_exp_name=complete exp_name=eval_complete \
    log_dir=/kaggle/working/dcp_results > /kaggle/working/eval_complete.log 2>&1

print("---- LAST 40 LINES ----")
!tail -40 /kaggle/working/eval_complete.log
print("\n---- RECORDS ----")
!cat records/hatememes/*complete* 2>/dev/null || echo "no complete record file"

In [ ]:
!ls -la /kaggle/working/dcpenv/bin/ | head -20
print("---")
!/kaggle/working/dcpenv/bin/python --version 2>&1 | head -3
print("---")
!ls -la /root/.local/share/uv/python/ 2>&1 | head -5

In [ ]:
import os
os.environ['UV_PYTHON_INSTALL_DIR'] = '/kaggle/working/uvpython'

!rm -rf /kaggle/working/dcpenv
!UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython uv venv --python 3.9 /kaggle/working/dcpenv

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  "numpy==1.23.5" "setuptools==65.6.3" \
  pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 \
  timm==0.6.13 sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 \
  "Pillow==9.5.0" einops ftfy jsonlines regex tqdm

print("\nCHECK:")
!/kaggle/working/dcpenv/bin/python -c "import torch, pytorch_lightning as pl; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| pl', pl.__version__)"

In [ ]:
import glob, os
%cd /kaggle/working/Deep_Correlated_Prompting

ckpts = [c for c in glob.glob('/kaggle/working/dcp_results/base_fixed_seed0/version_*/checkpoints/*.ckpt')
         if 'last' not in os.path.basename(c)]
BEST = sorted(ckpts)[-1]
print("BEST:", BEST)

!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes load_path={BEST} test_only=True \
    test_ratio=0.0 test_type=both test_exp_name=complete exp_name=eval_complete \
    log_dir=/kaggle/working/dcp_results > /kaggle/working/eval_complete.log 2>&1

print("---- LAST 30 LINES ----")
!tail -30 /kaggle/working/eval_complete.log
print("---- RECORD ----")
!cat records/hatememes/*complete* 2>/dev/null || echo "no record file"

In [ ]:
import glob, os
%cd /kaggle/working/Deep_Correlated_Prompting
ckpts = [c for c in glob.glob('/kaggle/working/dcp_results/base_fixed_seed0/version_*/checkpoints/*.ckpt')
         if 'last' not in os.path.basename(c)]
BEST = sorted(ckpts)[-1]

for t in ['text', 'image']:
    !/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={BEST} test_only=True \
        test_ratio=1.0 test_type={t} test_exp_name=only exp_name=eval_only_{t} \
        log_dir=/kaggle/working/dcp_results > /kaggle/working/only_{t}.log 2>&1

print("=== 100% text removed (image-only performance) ===")
!grep "AUROC_epoch" /kaggle/working/only_text.log
print("=== 100% image removed (text-only performance) ===")
!grep "AUROC_epoch" /kaggle/working/only_image.log

In [ ]:
import glob, os
%cd /kaggle/working/Deep_Correlated_Prompting
LAST = glob.glob('/kaggle/working/dcp_results/base_fixed_seed0/version_*/checkpoints/last.ckpt')[0]
print("LAST:", LAST)

for t in ['text', 'image']:
    !/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={LAST} test_only=True \
        test_ratio=1.0 test_type={t} exp_name=last_only_{t} \
        log_dir=/kaggle/working/dcp_results > /kaggle/working/last_{t}.log 2>&1

print("epoch19 | text removed (image-only):")
!grep "AUROC_epoch" /kaggle/working/last_text.log
print("epoch19 | image removed (text-only):")
!grep "AUROC_epoch" /kaggle/working/last_image.log

In [ ]:
import time
%cd /kaggle/working/Deep_Correlated_Prompting
start = time.time()
!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes exp_name=base_seed1 seed=1 \
    max_epoch=20 val_check_interval=1.0 \
    log_dir=/kaggle/working/dcp_results
print(f"\n=== TOTAL: {(time.time()-start)/60:.1f} MINUTES ===")

In [ ]:
import glob, os
%cd /kaggle/working/Deep_Correlated_Prompting

ckpts = [c for c in glob.glob('/kaggle/working/dcp_results/base_seed1_seed1/version_*/checkpoints/*.ckpt')
         if 'last' not in os.path.basename(c)]
print("found:", ckpts)
BEST1 = sorted(ckpts)[-1]
print("BEST (seed1):", BEST1)

for t in ['text', 'image']:
    !/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={BEST1} test_only=True \
        test_ratio=1.0 test_type={t} exp_name=s1_only_{t} \
        log_dir=/kaggle/working/dcp_results > /kaggle/working/s1_{t}.log 2>&1

print("\nseed1 | text removed (IMAGE-ONLY):")
!grep "AUROC_epoch" /kaggle/working/s1_text.log
print("seed1 | image removed (TEXT-ONLY):")
!grep "AUROC_epoch" /kaggle/working/s1_image.log

In [ ]:
import glob, os
%cd /kaggle/working/Deep_Correlated_Prompting
os.makedirs('records/hatememes', exist_ok=True)
BEST1 = '/kaggle/working/dcp_results/base_seed1_seed1/version_0/checkpoints/epoch=8-step=306.ckpt'

for t in ['image', 'text', 'both']:
    !/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={BEST1} test_only=True \
        test_ratio=0.7 test_type={t} test_exp_name=seed1 exp_name=s1_eval_{t} \
        log_dir=/kaggle/working/dcp_results > /kaggle/working/s1_eval_{t}.log 2>&1

for f in sorted(glob.glob('records/hatememes/*seed1*')):
    print(f.split('/')[-1], "->", open(f).read().strip())

In [ ]:
import pyarrow as pa, io
from PIL import Image
import matplotlib.pyplot as plt

t = pa.ipc.RecordBatchFileReader(
    pa.memory_map('/kaggle/working/dl/arrow/hatememes_test.arrow')).read_all()

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, i in zip(axes, [0, 1, 2, 3]):
    img = Image.open(io.BytesIO(t['image'][i].as_py())).convert('RGB')
    ax.imshow(img); ax.axis('off')
    ax.set_title(t['text'][i][0].as_py()[:40], fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
import os, glob
for p in ['/kaggle/working/dl/arrow', '/kaggle/working/Deep_Correlated_Prompting',
          '/kaggle/working/dcpenv/bin/python']:
    print(f"{p}: {'EXISTS' if os.path.exists(p) else 'MISSING'}")

c = [x for x in glob.glob('/kaggle/working/dcp_results/**/*.ckpt', recursive=True) if 'last' not in x]
print("\ncheckpoints:", c)

In [ ]:
BASE = '/kaggle/working/Deep_Correlated_Prompting'

p = f'{BASE}/clip/config.py'
s = open(p).read()
if 'fake_missing_type' not in s:
    s = s.replace("    test_ratio = None", "    test_ratio = None\n    fake_missing_type = None")
    open(p, 'w').write(s)

p = f'{BASE}/clip/modules/clip_missing_aware_prompt_module.py'
s = open(p).read()
anchor = '        both_feats = self.model(img, text, batch["missing_type"])'
patch = ('        _fmt = self.hparams.config["fake_missing_type"]\n'
         '        if _fmt is not None:\n'
         '            batch["missing_type"] = [(0 if int(v) == 0 else int(_fmt)) for v in batch["missing_type"]]\n'
         + anchor)
if '_fmt = self.hparams.config' not in s:
    s = s.replace(anchor, patch)
    open(p, 'w').write(s)

print("CONFIG:")
!grep -n "fake_missing_type" {BASE}/clip/config.py
print("MODULE:")
!grep -n "_fmt" {BASE}/clip/modules/clip_missing_aware_prompt_module.py

In [ ]:
import glob, os
!chmod -R +x /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting

BEST = '/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt'

runs = [('image', None, 'image missing, told correctly'),
        ('image', 1,    'image missing, told TEXT is missing   <-- LIE'),
        ('text',  None, 'text missing, told correctly'),
        ('text',  2,    'text missing, told IMAGE is missing   <-- LIE')]

for actual, fake, label in runs:
    tag = f"{actual}_{fake}"
    extra = f"fake_missing_type={fake}" if fake is not None else ""
    !/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={BEST} test_only=True \
        test_ratio=0.7 test_type={actual} {extra} exp_name=lie_{tag} \
        log_dir=/kaggle/working/dcp_results > /kaggle/working/lie_{tag}.log 2>&1
    print(f"\n{label}")
    !grep "AUROC_epoch" /kaggle/working/lie_{tag}.log

In [ ]:
!tail -30 /kaggle/working/lie_image_None.log
print("=" * 60)
!tail -15 /kaggle/working/lie_image_1.log

In [ ]:
!chmod -R +x /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting
!ls -la /kaggle/working/lie_*.log

!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes \
    load_path=/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt \
    test_only=True test_ratio=0.7 test_type=image fake_missing_type=1 \
    exp_name=lie_test log_dir=/kaggle/working/dcp_results

In [ ]:
!/kaggle/working/dcpenv/bin/python -c "print('hello')"
!/kaggle/working/dcpenv/bin/python -c "import torch; print('torch ok', torch.cuda.is_available())"

In [ ]:
import os
print("dcpenv/bin/python ->", os.path.realpath('/kaggle/working/dcpenv/bin/python'))
print("target exists:", os.path.exists(os.path.realpath('/kaggle/working/dcpenv/bin/python')))
print("uvpython dir exists:", os.path.exists('/kaggle/working/uvpython'))

os.environ['UV_PYTHON_INSTALL_DIR'] = '/kaggle/working/uvpython'
!pip install -q uv
!rm -rf /kaggle/working/dcpenv
!UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython uv venv --python 3.9 /kaggle/working/dcpenv

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  "numpy==1.23.5" "setuptools==65.6.3" \
  pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 \
  timm==0.6.13 sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 \
  "Pillow==9.5.0" einops ftfy jsonlines regex tqdm

!chmod -R +x /kaggle/working/dcpenv/bin
print("\nCHECK:")
!/kaggle/working/dcpenv/bin/python -c "import torch, pytorch_lightning as pl; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| pl', pl.__version__)"

In [ ]:
import os
os.environ['UV_PYTHON_INSTALL_DIR'] = '/kaggle/working/uvpython'

!chmod -R +x /kaggle/working/uvpython

!UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython uv venv --python 3.9 /kaggle/working/dcpenv

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  "numpy==1.23.5" "setuptools==65.6.3" \
  pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 \
  timm==0.6.13 sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 \
  "Pillow==9.5.0" einops ftfy jsonlines regex tqdm

!chmod -R +x /kaggle/working/dcpenv/bin
print("\nCHECK:")
!/kaggle/working/dcpenv/bin/python -c "import torch, pytorch_lightning as pl; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| pl', pl.__version__)"

In [ ]:
import glob, os
!chmod -R +x /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting

BEST = '/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt'

runs = [('image', None, 'image missing, told correctly'),
        ('image', 1,    'image missing, told TEXT is missing   <-- LIE'),
        ('text',  None, 'text missing, told correctly'),
        ('text',  2,    'text missing, told IMAGE is missing   <-- LIE')]

for actual, fake, label in runs:
    tag = f"{actual}_{fake}"
    extra = f"fake_missing_type={fake}" if fake is not None else ""
    !/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={BEST} test_only=True \
        test_ratio=0.7 test_type={actual} {extra} exp_name=lie_{tag} \
        log_dir=/kaggle/working/dcp_results > /kaggle/working/lie_{tag}.log 2>&1
    print(f"\n{label}")
    !grep "AUROC_epoch" /kaggle/working/lie_{tag}.log

In [ ]:
!chmod -R +x /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting

!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes \
    load_path=/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt \
    test_only=True test_ratio=0.7 test_type=image fake_missing_type=1 \
    exp_name=lie_direct log_dir=/kaggle/working/dcp_results

In [ ]:
print("1) binary works?")
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

print("\n2) patched file syntax ok?")
!/kaggle/working/dcpenv/bin/python -m py_compile /kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py && echo "syntax OK"

print("\n3) config file syntax ok?")
!/kaggle/working/dcpenv/bin/python -m py_compile /kaggle/working/Deep_Correlated_Prompting/clip/config.py && echo "syntax OK"

print("\n4) free memory:")
!free -g | head -2

print("\n5) the patched lines:")
!sed -n '240,248p' /kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py

In [ ]:
!/kaggle/working/dcpenv/bin/python -c "print('hello')" > /tmp/t.txt 2>&1 ; cat /tmp/t.txt ; wc -c /tmp/t.txt

In [ ]:
import os
os.environ['UV_PYTHON_INSTALL_DIR'] = '/kaggle/working/uvpython'

!pip install -q uv
!rm -rf /kaggle/working/dcpenv /kaggle/working/uvpython
!UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython uv venv --python 3.9 /kaggle/working/dcpenv

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

!uv pip install --python /kaggle/working/dcpenv/bin/python \
  "numpy==1.23.5" "setuptools==65.6.3" \
  pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 \
  timm==0.6.13 sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 \
  "Pillow==9.5.0" einops ftfy jsonlines regex tqdm

!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
print("TEST:")
!/kaggle/working/dcpenv/bin/python -c "print('hello'); import torch; print('torch', torch.__version__, torch.cuda.is_available())"

In [ ]:
!chmod -R +x /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting

BEST = '/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt'

runs = [('image', None, 'image missing, told correctly'),
        ('image', 1,    'image missing, told TEXT is missing   <-- LIE'),
        ('text',  None, 'text missing, told correctly'),
        ('text',  2,    'text missing, told IMAGE is missing   <-- LIE')]

for actual, fake, label in runs:
    tag = f"{actual}_{fake}"
    extra = f"fake_missing_type={fake}" if fake is not None else ""
    print(f"\n===== {label} =====")
    !/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={BEST} test_only=True \
        test_ratio=0.7 test_type={actual} {extra} exp_name=lie_{tag} \
        log_dir=/kaggle/working/dcp_results 2>&1 | grep -E "AUROC_epoch|accuracy_epoch|Error|error"

In [ ]:
%cd /kaggle/working/Deep_Correlated_Prompting
BEST = '/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt'
!/kaggle/working/dcpenv/bin/python run.py with data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes load_path={BEST} test_only=True \
    test_ratio=0.7 test_type=image exp_name=lie_image_none2 \
    log_dir=/kaggle/working/dcp_results 2>&1 | grep -E "AUROC_epoch|accuracy_epoch"

In [ ]:
%%writefile /kaggle/working/build_emb.py
import pyarrow as pa, io, torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

dev = 'cuda'
m = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(dev).eval()
p = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

@torch.no_grad()
def emb(split, bs=64):
    t = pa.ipc.RecordBatchFileReader(
        pa.memory_map(f'/kaggle/working/dl/arrow/hatememes_{split}.arrow', 'r')).read_all()
    n = t.num_rows
    I, T = [], []
    for s in range(0, n, bs):
        e = min(s + bs, n)
        imgs = [Image.open(io.BytesIO(t['image'][i].as_py())).convert('RGB') for i in range(s, e)]
        txts = [t['text'][i][0].as_py() for i in range(s, e)]
        f = m.get_image_features(**p(images=imgs, return_tensors='pt').to(dev))
        I.append((f / f.norm(dim=-1, keepdim=True)).cpu())
        f = m.get_text_features(**p(text=txts, return_tensors='pt', padding=True,
                                    truncation=True, max_length=77).to(dev))
        T.append((f / f.norm(dim=-1, keepdim=True)).cpu())
    return torch.cat(I), torch.cat(T)

out = {}
for sp in ['train', 'dev', 'test']:
    i, tt = emb(sp)
    out[sp] = {'image': i, 'text': tt}
    print(sp, 'image', tuple(i.shape), 'text', tuple(tt.shape))
torch.save(out, '/kaggle/working/clip_emb.pt')
print("saved")

In [ ]:
!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python /kaggle/working/build_emb.py

In [ ]:
#40
P = '/kaggle/working/Deep_Correlated_Prompting/clip/datasets/hatememes_dataset.py'
s = open(P).read()

if 'DCP_RETRIEVAL' not in s:
    s = s.replace("import torch\nimport random, os",
                  "import torch\nimport random, os, io\nimport pyarrow as pa\nfrom PIL import Image")

    s = s.replace("""        self.missing_table = missing_table""",
"""        self.missing_table = missing_table

        # ---- retrieval (our contribution) ----
        self.retrieval = os.environ.get('DCP_RETRIEVAL', '0') == '1'
        self.retrieval_k = int(os.environ.get('DCP_RETRIEVAL_K', '1'))
        if self.retrieval:
            E = torch.load('/kaggle/working/clip_emb.pt')
            key = {'train': 'train', 'val': 'dev', 'test': 'test'}[split]
            self.q_img = E[key]['image']
            self.q_txt = E[key]['text']
            self.pool_img = E['train']['image']
            self.pool_txt = E['train']['text']
            self.pool_table = pa.ipc.RecordBatchFileReader(
                pa.memory_map('/kaggle/working/dl/arrow/hatememes_train.arrow', 'r')).read_all()
            self.is_train_split = (split == 'train')

    def _topk(self, query, pool, own_index):
        sims = pool @ query
        if self.is_train_split:
            sims[own_index] = -1e9          # never retrieve yourself
        return torch.topk(sims, self.retrieval_k).indices.tolist()""")

    s = s.replace("""        # missing image, dummy image is all-one image
        if self.missing_table[image_index] == 2 or simulate_missing_type == 2:
            for idx in range(len(image_tensor)):
                image_tensor[idx] = torch.ones(image_tensor[idx].size()).float()""",
"""        # missing image
        if self.missing_table[image_index] == 2 or simulate_missing_type == 2:
            if self.retrieval:
                js = self._topk(self.q_txt[image_index], self.pool_img, image_index)
                stack = []
                for j in js:
                    rimg = Image.open(io.BytesIO(self.pool_table['image'][j].as_py())).convert('RGB')
                    stack.append([tr(rimg) for tr in self.transforms])
                image_tensor = [torch.stack([s[k] for s in stack]).mean(0)
                                for k in range(len(stack[0]))]
            else:
                for idx in range(len(image_tensor)):
                    image_tensor[idx] = torch.ones(image_tensor[idx].size()).float()""")

    s = s.replace("""        #missing text, dummy text is ''
        if self.missing_table[image_index] == 1 or simulate_missing_type == 1:
            text = ''""",
"""        # missing text
        if self.missing_table[image_index] == 1 or simulate_missing_type == 1:
            if self.retrieval:
                js = self._topk(self.q_img[image_index], self.pool_txt, image_index)
                text = ' '.join(self.pool_table['text'][j][0].as_py() for j in js)
            else:
                text = ''""")

    open(P, 'w').write(s)

print("PATCH CHECK:")
!grep -n "DCP_RETRIEVAL\|_topk\|never retrieve yourself" {P}
!/kaggle/working/dcpenv/bin/python -m py_compile {P} && echo "syntax OK"

In [ ]:
import os
assert os.path.exists('/kaggle/working/clip_emb.pt'), "pehle CELL 1 (build_emb.py) chalao"

!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting

BEST = '/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt'

for t in ['image', 'text']:
    for retr in ['0', '1']:
        name = 'RETRIEVAL' if retr == '1' else 'baseline (dummy)'
        print(f"\n===== missing {t}  |  {name} =====")
        !DCP_RETRIEVAL={retr} DCP_RETRIEVAL_K=1 /kaggle/working/dcpenv/bin/python run.py with \
            data_root=/kaggle/working/dl/arrow \
            num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
            task_finetune_hatememes load_path={BEST} test_only=True \
            test_ratio=0.7 test_type={t} exp_name=retr_{t}_{retr} \
            log_dir=/kaggle/working/dcp_results 2>&1 | grep -E "AUROC_epoch|accuracy_epoch|Error"

In [ ]:
!DCP_RETRIEVAL=1 /kaggle/working/dcpenv/bin/python -c "import os; print('env =', os.environ.get('DCP_RETRIEVAL'))"

In [ ]:
P = '/kaggle/working/Deep_Correlated_Prompting/clip/datasets/hatememes_dataset.py'
s = open(P).read()
if 'RETRIEVAL IS ON' not in s:
    s = s.replace("            self.is_train_split = (split == 'train')",
"""            self.is_train_split = (split == 'train')
            print(f">>> RETRIEVAL IS ON | split={split} | k={self.retrieval_k}", flush=True)""")
    open(P, 'w').write(s)
!grep -n "RETRIEVAL IS ON" {P}

%cd /kaggle/working/Deep_Correlated_Prompting
BEST = '/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt'
!DCP_RETRIEVAL=1 DCP_RETRIEVAL_K=1 /kaggle/working/dcpenv/bin/python run.py with \
    data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes load_path={BEST} test_only=True \
    test_ratio=0.7 test_type=image exp_name=retr_check \
    log_dir=/kaggle/working/dcp_results 2>&1 | grep -E "RETRIEVAL IS ON|AUROC_epoch"

In [ ]:
P = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
s = open(P).read()

if '\nimport os' not in s[:600]:
    s = s.replace('import torch\n', 'import os\nimport torch\n', 1)

old = """        for idx in range(len(img)):
            if batch["missing_type"][idx] == 0:
                pass       
            elif batch["missing_type"][idx] == 1:  # missing text
                both_feats[idx, feature_dim:].zero_()
            elif batch["missing_type"][idx] == 2:
                both_feats[idx, :feature_dim].zero_()"""

new = """        _retr = os.environ.get('DCP_RETRIEVAL', '0') == '1'
        for idx in range(len(img)):
            if batch["missing_type"][idx] == 0:
                pass       
            elif batch["missing_type"][idx] == 1:  # missing text
                if not _retr:
                    both_feats[idx, feature_dim:].zero_()
            elif batch["missing_type"][idx] == 2:
                if not _retr:
                    both_feats[idx, :feature_dim].zero_()"""

if '_retr = os.environ' not in s:
    assert old in s, "block not found"
    s = s.replace(old, new)
    open(P, 'w').write(s)

!grep -n "_retr" {P}
!/kaggle/working/dcpenv/bin/python -m py_compile {P} && echo "syntax OK"

In [ ]:
%cd /kaggle/working/Deep_Correlated_Prompting
BEST = '/kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt'

for t in ['image', 'text']:
    for retr in ['0', '1']:
        name = 'RETRIEVAL' if retr == '1' else 'baseline (dummy)'
        print(f"\n===== missing {t}  |  {name} =====")
        !DCP_RETRIEVAL={retr} DCP_RETRIEVAL_K=1 /kaggle/working/dcpenv/bin/python run.py with \
            data_root=/kaggle/working/dl/arrow \
            num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
            task_finetune_hatememes load_path={BEST} test_only=True \
            test_ratio=0.7 test_type={t} exp_name=r2_{t}_{retr} \
            log_dir=/kaggle/working/dcp_results 2>&1 | grep -E "AUROC_epoch|accuracy_epoch"

In [ ]:
import time
%cd /kaggle/working/Deep_Correlated_Prompting
start = time.time()
!DCP_RETRIEVAL=1 DCP_RETRIEVAL_K=1 /kaggle/working/dcpenv/bin/python run.py with \
    data_root=/kaggle/working/dl/arrow \
    num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
    task_finetune_hatememes exp_name=retr_train \
    max_epoch=20 val_check_interval=1.0 \
    log_dir=/kaggle/working/dcp_results
print(f"\n=== TOTAL: {(time.time()-start)/60:.1f} MINUTES ===")

In [ ]:
import glob, os, pandas as pd
!chmod -R +x /kaggle/working/dcpenv/bin
%cd /kaggle/working/Deep_Correlated_Prompting

# sanity check pehle
p = sorted(glob.glob('/kaggle/working/dcp_results/retr_train_seed0/version_*/metrics.csv'))
df = pd.read_csv(p[-1])
print("max lr (must be > 0):", float(df[[c for c in df.columns if 'lr-' in c]].max().max()))
tl = df['hatememes/train/loss'].dropna()
print(f"train loss: {tl.iloc[0]:.4f} -> {tl.iloc[-1]:.4f}")

ck = [c for c in glob.glob('/kaggle/working/dcp_results/retr_train_seed0/version_*/checkpoints/*.ckpt')
      if 'last' not in os.path.basename(c)]
BEST_R = sorted(ck)[-1]
print("retrieval checkpoint:", BEST_R)

for t in ['image', 'text', 'both']:
    print(f"\n===== missing {t}  |  RETRIEVAL (trained) =====")
    !DCP_RETRIEVAL=1 DCP_RETRIEVAL_K=1 /kaggle/working/dcpenv/bin/python run.py with \
        data_root=/kaggle/working/dl/arrow \
        num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 \
        task_finetune_hatememes load_path={BEST_R} test_only=True \
        test_ratio=0.7 test_type={t} exp_name=rtr_eval_{t} \
        log_dir=/kaggle/working/dcp_results 2>&1 | grep -E "AUROC_epoch|accuracy_epoch"

In [ ]:

import torch, pyarrow as pa
E = torch.load('/kaggle/working/clip_emb.pt')

def labels(split):
    t = pa.ipc.RecordBatchFileReader(
        pa.memory_map(f'/kaggle/working/dl/arrow/hatememes_{split}.arrow','r')).read_all()
    return torch.tensor([t['label'][i].as_py() for i in range(t.num_rows)])

tr_lab, te_lab = labels('train'), labels('test')
sims = E['test']['text'] @ E['train']['image'].T      # text query -> image pool
j = sims.argmax(1)
agree = (tr_lab[j] == te_lab).float().mean()

print(f"retrieved-image label matches query label: {100*agree:.1f}% of the time")
print(f"chance level (train class balance): {100*max(tr_lab.float().mean(), 1-tr_lab.float().mean()):.1f}%")
print(f"mean cosine similarity of the retrieved match: {sims.max(1).values.mean():.3f}")


In [ ]:
import torch, pyarrow as pa

E = torch.load('/kaggle/working/clip_emb.pt')

def labels(split):
    t = pa.ipc.RecordBatchFileReader(
        pa.memory_map(f'/kaggle/working/dl/arrow/hatememes_{split}.arrow','r')).read_all()
    return torch.tensor([t['label'][i].as_py() for i in range(t.num_rows)])

tr_lab, te_lab = labels('train'), labels('test')

sims = E['test']['text'] @ E['train']['image'].T
j = sims.argmax(1)
agree = (tr_lab[j] == te_lab).float().mean()

print(f"retrieved-image label matches query label : {100*agree:.1f}%")
print(f"chance level (train class balance)        : {100*max(tr_lab.float().mean(), 1-tr_lab.float().mean()):.1f}%")
print(f"mean cosine similarity of retrieved match : {sims.max(1).values.mean():.3f}")

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
src = open(p).read()
i = src.index('def infer(')
j = src.index('\n    def ', i+10)
print(src[i:j])

In [ ]:
base = '/kaggle/working/Deep_Correlated_Prompting/clip'
s = open(base + '/modules/clip_missing_aware_prompt_module.py').read()
lines = s.split('\n')
print("----- CLASSIFIER BUILD -----")
for n, l in enumerate(lines):
    if 'hatememes_classifier' in l and 'nn.' in l or ('hatememes_classifier =' in l):
        print('\n'.join(lines[max(0, n-3):n+12]))
        break
print("\n----- TRAINING STEP -----")
i = s.index('def training_step')
print(s[i:i+800])
print("\n----- COMPUTE_HATEMEMES -----")
o = open(base + '/modules/objectives.py').read()
j = o.index('def compute_hatememes')
print(o[j:j+1400])

In [ ]:
base = '/kaggle/working/Deep_Correlated_Prompting/clip'
s = open(base + '/modules/clip_missing_aware_prompt_module.py').read()
lines = s.split('\n')
print("----- HIDDEN_SIZE AND FREEZING -----")
for n, l in enumerate(lines):
    if 'hidden_size' in l or 'requires_grad' in l:
        print(n, l)
print("\n----- SET_SCHEDULE -----")
u = open(base + '/modules/clip_utils.py').read()
i = u.index('def set_schedule')
print(u[i:i+2500])

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
lines = open(p).read().split('\n')
print('\n'.join(f"{n} {l}" for n, l in enumerate(lines[196:226], start=196)))

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
src = open(p).read()

anchor = "self.mmimdb_classifier.apply(objectives.init_weights)"
assert anchor in src, "anchor not found in __init__"
if "self.gen_i2t" not in src:
    ins = """

        if os.environ.get('DCP_GEN', '0') == '1':
            gdim = hidden_size // 2
            self.gen_i2t = nn.Sequential(nn.Linear(gdim, gdim), nn.ReLU(), nn.Linear(gdim, gdim))
            self.gen_t2i = nn.Sequential(nn.Linear(gdim, gdim), nn.ReLU(), nn.Linear(gdim, gdim))
            print(">>> GENERATORS CREATED | dim=" + str(gdim), flush=True)
"""
    k = src.index(anchor) + len(anchor)
    src = src[:k] + ins + src[k:]

a = src.index("        _retr = os.environ.get")
b = src.index("        ret = {", a)
new_block = '''        _retr = os.environ.get('DCP_RETRIEVAL', '0') == '1'
        _gen = os.environ.get('DCP_GEN', '0') == '1'
        self._recon_loss = None
        if _gen:
            _lam = float(os.environ.get('DCP_GEN_LAMBDA', '1.0'))
            mt = torch.tensor([int(v) for v in batch["missing_type"]], device=both_feats.device)
            img_half = both_feats[:, :feature_dim]
            txt_half = both_feats[:, feature_dim:]
            gen_txt = self.gen_i2t(img_half.float()).to(both_feats.dtype)
            gen_img = self.gen_t2i(txt_half.float()).to(both_feats.dtype)
            m1 = (mt == 1).unsqueeze(1).to(both_feats.dtype)
            m2 = (mt == 2).unsqueeze(1).to(both_feats.dtype)
            new_img = img_half * (1 - m2) + gen_img * m2
            new_txt = txt_half * (1 - m1) + gen_txt * m1
            both_feats = torch.cat([new_img, new_txt], dim=1)
            m0 = (mt == 0)
            if self.training and bool(m0.any()):
                r1 = torch.nn.functional.mse_loss(gen_txt[m0].float(), txt_half[m0].float().detach())
                r2 = torch.nn.functional.mse_loss(gen_img[m0].float(), img_half[m0].float().detach())
                self._recon_loss = _lam * (r1 + r2)
        else:
            for idx in range(len(img)):
                if batch["missing_type"][idx] == 0:
                    pass
                elif batch["missing_type"][idx] == 1:
                    if not _retr:
                        both_feats[idx, feature_dim:].zero_()
                elif batch["missing_type"][idx] == 2:
                    if not _retr:
                        both_feats[idx, :feature_dim].zero_()

'''
src = src[:a] + new_block + src[b:]
open(p, 'w').write(src)

o_path = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/objectives.py'
o = open(o_path).read()
tgt = '''        "hatememes_labels": imgcls_labels,
    }
'''
assert tgt in o, "anchor not found in objectives.py"
if "_recon_loss" not in o:
    add = '''    if getattr(pl_module, "_recon_loss", None) is not None:
        ret["recon_loss"] = pl_module._recon_loss
'''
    k = o.index(tgt) + len(tgt)
    o = o[:k] + add + o[k:]
    open(o_path, 'w').write(o)

print("PATCH APPLIED")
i = src.index('def infer(')
j = src.index('\n    def ', i + 10)
print(src[i:j])

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
src = open(p).read()
old = "                self._recon_loss = _lam * (r1 + r2)\n"
assert old in src, "anchor not found"
if "_gen_dbg" not in src:
    new = old + """                self._gen_dbg = getattr(self, '_gen_dbg', 0) + 1
                if self._gen_dbg % 100 == 1:
                    print(">>> GEN step=" + str(self._gen_dbg) + " recon=" + str(round(float(self._recon_loss), 4)) + " complete_in_batch=" + str(int(m0.sum())), flush=True)
"""
    src = src.replace(old, new)
    open(p, 'w').write(src)
print("DEBUG PRINT ADDED")

In [ ]:
import glob, os
for f in sorted(glob.glob('/kaggle/working/**/*.ckpt', recursive=True)):
    print(round(os.path.getsize(f)/1e6, 1), "MB", f)
print("---")
for d in sorted(glob.glob('/kaggle/working/*')):
    print(os.path.basename(d))

In [ ]:
import os, glob
d = '/kaggle/working/dcp_results/retr_train_seed0/version_0'
print(os.listdir(d))
for f in glob.glob(d + '/hparams.yaml'):
    print(open(f).read()[:3000])

In [ ]:
import re
s = open('/kaggle/working/Deep_Correlated_Prompting/clip/config.py').read()
print(re.findall(r'@ex\.named_config\s*\r?\ndef (\w+)', s))

In [ ]:
!cd /kaggle/working/Deep_Correlated_Prompting && \
DCP_GEN=1 DCP_GEN_LAMBDA=1.0 DCP_RETRIEVAL=0 \
/kaggle/working/dcpenv/bin/python -u run.py with \
data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 \
task_finetune_hatememes per_gpu_batchsize=4 batch_size=256 \
max_epoch=20 learning_rate=1e-2 seed=0 exp_name=gen_seed0 \
log_dir=/kaggle/working/dcp_results num_workers=4 val_check_interval=1.0 \
prompt_length=36 prompt_depth=6 both_ratio=0.5 \
"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" \
"missing_type.train=both" "missing_type.val=both" "missing_type.test=both" \
2>&1 | tee /kaggle/working/gen_seed0.log

In [ ]:
!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

In [ ]:
import shutil, os
shutil.rmtree('/kaggle/working/dcpenv', ignore_errors=True)
shutil.rmtree('/kaggle/working/uvpython', ignore_errors=True)
print("OLD ENV REMOVED")

In [ ]:
%env UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
!pip install -q uv
!uv venv --python 3.9 /kaggle/working/dcpenv
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

In [ ]:
P = '/kaggle/working/dcpenv/bin/python'
!{P} -m pip install -q --upgrade pip
!{P} -m pip install -q numpy==1.23.5 setuptools==65.6.3
!{P} -m pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117
!{P} -m pip install -q pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 timm==0.6.13
!{P} -m pip install -q sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 Pillow==9.5.0 einops ftfy jsonlines regex tqdm
print("INSTALL DONE")
!{P} -c "import torch, pytorch_lightning, sacred, pyarrow; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('pl', pytorch_lightning.__version__)"

In [ ]:
P = '/kaggle/working/dcpenv/bin/python'
!uv pip install --python {P} -q numpy==1.23.5 setuptools==65.6.3 wheel
!uv pip install --python {P} -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117
!uv pip install --python {P} -q pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 timm==0.6.13
!uv pip install --python {P} -q sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 Pillow==9.5.0 einops ftfy jsonlines regex tqdm
print("INSTALL DONE")
!{P} -c "import torch, pytorch_lightning; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('pl', pytorch_lightning.__version__)"

In [ ]:
!cd /kaggle/working/Deep_Correlated_Prompting && \
DCP_GEN=1 DCP_GEN_LAMBDA=1.0 DCP_RETRIEVAL=0 \
/kaggle/working/dcpenv/bin/python -u run.py with \
data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 \
task_finetune_hatememes per_gpu_batchsize=4 batch_size=256 \
max_epoch=20 learning_rate=1e-2 seed=0 exp_name=gen_seed0 \
log_dir=/kaggle/working/dcp_results num_workers=4 val_check_interval=1.0 \
prompt_length=36 prompt_depth=6 both_ratio=0.5 \
"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" \
"missing_type.train=both" "missing_type.val=both" "missing_type.test=both" \
2>&1 | tee /kaggle/working/gen_seed0.log

In [ ]:
import glob, os
for f in sorted(glob.glob('/kaggle/working/dcp_results/**/*.ckpt', recursive=True)):
    print(f)

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/working/dcp_results/gen_seed0_seed0/version_6/metrics.csv')
cols = [c for c in df.columns if 'val' in c or 'train' in c or c == 'epoch' or 'lr' in c]
d = df[cols].groupby(df['epoch']).max()
pd.set_option('display.width', 200)
print(d.to_string())

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
src = open(p).read()
a = src.index("        if _gen:")
b = src.index("        else:\n            for idx in range(len(img)):")
new = '''        if _gen:
            _lam = float(os.environ.get('DCP_GEN_LAMBDA', '1.0'))
            F_ = torch.nn.functional
            mt = torch.tensor([int(v) for v in batch["missing_type"]], device=both_feats.device)
            img_half = both_feats[:, :feature_dim]
            txt_half = both_feats[:, feature_dim:]
            m0 = (mt == 0)
            if bool(m0.any()):
                n_txt = txt_half[m0].float().norm(dim=1).mean().detach()
                n_img = img_half[m0].float().norm(dim=1).mean().detach()
            else:
                n_txt = txt_half.float().norm(dim=1).mean().detach()
                n_img = img_half.float().norm(dim=1).mean().detach()
            gen_txt = (F_.normalize(self.gen_i2t(img_half.float()), dim=1) * n_txt).to(both_feats.dtype)
            gen_img = (F_.normalize(self.gen_t2i(txt_half.float()), dim=1) * n_img).to(both_feats.dtype)
            m1 = (mt == 1).unsqueeze(1).to(both_feats.dtype)
            m2 = (mt == 2).unsqueeze(1).to(both_feats.dtype)
            new_img = img_half * (1 - m2) + gen_img * m2
            new_txt = txt_half * (1 - m1) + gen_txt * m1
            both_feats = torch.cat([new_img, new_txt], dim=1)
            if self.training and bool(m0.any()):
                c1 = 1 - F_.cosine_similarity(gen_txt[m0].float(), txt_half[m0].float().detach(), dim=1).mean()
                c2 = 1 - F_.cosine_similarity(gen_img[m0].float(), img_half[m0].float().detach(), dim=1).mean()
                self._recon_loss = _lam * (c1 + c2)
                self._gen_dbg = getattr(self, '_gen_dbg', 0) + 1
                if self._gen_dbg % 100 == 1:
                    print(">>> GEN step=" + str(self._gen_dbg) + " recon=" + str(round(float(self._recon_loss), 4)) + " featnorm=" + str(round(float(n_txt), 2)), flush=True)
'''
src = src[:a] + new + src[b:]
open(p, 'w').write(src)
print("PATCH 1 APPLIED")

u_path = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_utils.py'
u = open(u_path).read()
if 'DCP_GEN_LR_MULT' not in u:
    if 'import os' not in u.split('\n')[0] and '\nimport os' not in u:
        u = 'import os\n' + u
    anchor = '    if optim_type == "adamw":'
    assert anchor in u, "anchor not found in clip_utils"
    ins = '''    _gen_params = [p for n, p in pl_module.named_parameters() if n.startswith("gen_")]
    if len(_gen_params) > 0:
        _gen_ids = set(id(p) for p in _gen_params)
        for _g in optimizer_grouped_parameters:
            _g["params"] = [p for p in _g["params"] if id(p) not in _gen_ids]
        _glr = lr * float(os.environ.get("DCP_GEN_LR_MULT", "0.1"))
        optimizer_grouped_parameters.append({"params": _gen_params, "weight_decay": wd, "lr": _glr})
        print(">>> GEN PARAM GROUP lr=" + str(_glr), flush=True)

'''
    u = u.replace(anchor, ins + anchor, 1)
    open(u_path, 'w').write(u)
print("PATCH 2 APPLIED")

In [ ]:
!cd /kaggle/working/Deep_Correlated_Prompting && \
DCP_GEN=1 DCP_GEN_LAMBDA=1.0 DCP_GEN_LR_MULT=0.1 DCP_RETRIEVAL=0 \
/kaggle/working/dcpenv/bin/python -u run.py with \
data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 \
task_finetune_hatememes per_gpu_batchsize=4 batch_size=256 \
max_epoch=2 learning_rate=1e-2 seed=0 exp_name=gen2_smoke \
log_dir=/kaggle/working/dcp_results num_workers=4 val_check_interval=1.0 \
prompt_length=36 prompt_depth=6 both_ratio=0.5 \
"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" \
"missing_type.train=both" "missing_type.val=both" "missing_type.test=both" \
2>&1 | tee /kaggle/working/gen2_smoke.log

In [ ]:
import pandas as pd, glob
d = sorted(glob.glob('/kaggle/working/dcp_results/gen2_smoke_seed0/version_*/metrics.csv'))[-1]
df = pd.read_csv(d)
cols = [c for c in df.columns if 'val/AUROC' in c or 'val' in c or c == 'epoch']
print(df[cols].groupby(df['epoch']).max().to_string())

In [ ]:
import pandas as pd, glob
d = sorted(glob.glob('/kaggle/working/dcp_results/base_fixed_seed0/version_*/metrics.csv'))[-1]
df = pd.read_csv(d)
g = df.groupby('epoch').max()
print(g[['hatememes/val/AUROC_epoch', 'hatememes/val/loss_epoch']].head(10).to_string())

In [ ]:
!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

In [ ]:
import os, glob
B = '/kaggle/working/Deep_Correlated_Prompting'
m = open(f'{B}/clip/modules/clip_missing_aware_prompt_module.py').read()
u = open(f'{B}/clip/modules/clip_utils.py').read()
d = open(f'{B}/clip/datasets/hatememes_dataset.py').read()
c = open(f'{B}/clip/config.py').read()
print("max_steps fix     :", "max_steps == -1" in u)
print("fake_missing_type :", "fake_missing_type" in c and "fake_missing_type" in m)
print("retrieval         :", "DCP_RETRIEVAL" in d and "DCP_RETRIEVAL" in m)
print("generators        :", "gen_i2t" in m)
print("cosine + normalize:", "cosine_similarity" in m and "normalize" in m)
print("gen lr group      :", "DCP_GEN_LR_MULT" in u)
print("arrow / emb / venv:", os.path.exists('/kaggle/working/dl/arrow'),
      os.path.exists('/kaggle/working/clip_emb.pt'),
      os.path.exists('/kaggle/working/dcpenv/bin/python'))
print("checkpoints:", len([f for f in glob.glob('/kaggle/working/dcp_results/**/*.ckpt', recursive=True) if 'last' not in f]))

In [3]:
import os, glob

B = '/kaggle/working/Deep_Correlated_Prompting'
m = open(B + '/clip/modules/clip_missing_aware_prompt_module.py').read()
u = open(B + '/clip/modules/clip_utils.py').read()
d = open(B + '/clip/datasets/hatememes_dataset.py').read()
c = open(B + '/clip/config.py').read()
o = open(B + '/clip/modules/objectives.py').read()

print("PATCHES")
print("  max_steps fix     :", "max_steps == -1" in u)
print("  fake_missing_type :", "fake_missing_type" in c and "fake_missing_type" in m)
print("  retrieval         :", "DCP_RETRIEVAL" in d and "DCP_RETRIEVAL" in m)
print("  generators        :", "gen_i2t" in m)
print("  gen lr group      :", "DCP_GEN_LR_MULT" in u)
print("  detector          :", "DCP_DETECT" in m)
print("  modality gate     :", "DCP_MGATE" in m)
print("  logit dumping     :", "DUMPED-TEST" in m)
print("  eval on dev       :", "DCP_EVAL_ON_DEV" in d)

print()
print("HM ASSETS")
for f in ['hatememes_train.arrow', 'hatememes_dev.arrow', 'hatememes_test.arrow']:
    p = '/kaggle/working/dl/arrow/' + f
    print("  ", f, os.path.exists(p), round(os.path.getsize(p) / 1e6, 1) if os.path.exists(p) else 0, "MB")
print("   clip_emb.pt:", os.path.exists('/kaggle/working/clip_emb.pt'))
print("   venv python:", os.path.exists('/kaggle/working/dcpenv/bin/python'))

print()
print("HM DUMPS:")
hm = [f for f in os.listdir('/kaggle/working/dumps') if 'hm' in f or 'hate' in f]
print("  ", hm if hm else "NONE")

print()
print("HM CHECKPOINTS:")
found = 0
for p in sorted(glob.glob('/kaggle/working/dcp_results/*/version_*/checkpoints/*.ckpt')):
    n = p.split('/dcp_results/')[1].split('/')[0]
    if not any(k in n for k in ['mmimdb', 'mgate', 'retr', 'gen']):
        print("  ", n, "|", os.path.basename(p), round(os.path.getsize(p) / 1e6, 1), "MB")
        found += 1
print("   total HM checkpoints:", found)

PATCHES
  max_steps fix     : True
  fake_missing_type : True
  retrieval         : True
  generators        : True
  gen lr group      : True
  detector          : True
  modality gate     : True
  logit dumping     : True
  eval on dev       : True

HM ASSETS
   hatememes_train.arrow True 194.5 MB
   hatememes_dev.arrow True 189.1 MB
   hatememes_test.arrow True 195.5 MB
   clip_emb.pt: True
   venv python: True

HM DUMPS:
   NONE

HM CHECKPOINTS:
   mg_smoke_seed0 | epoch=0-step=12.ckpt 673.5 MB
   mg_smoke_seed0 | last.ckpt 673.5 MB
   total HM checkpoints: 2


In [2]:
import os, stat, glob

U = glob.glob('/kaggle/working/uvpython/cpython-3.9*/bin/python3.9')[0]
B = '/kaggle/working/dcpenv/bin'

print("REAL INTERPRETER:", U, round(os.path.getsize(U) / 1e6, 1), "MB")

for root, dirs, files in os.walk('/kaggle/working/uvpython'):
    for f in files:
        fp = os.path.join(root, f)
        if not os.path.islink(fp):
            try:
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
            except Exception:
                pass

for n in ['python', 'python3', 'python3.9']:
    t = os.path.join(B, n)
    if os.path.lexists(t):
        os.remove(t)
    os.symlink(U, t)
print("LINKED:", os.listdir(B)[:3])

open('/kaggle/working/dcpenv/pyvenv.cfg', 'w').write(
    "home = " + os.path.dirname(U) + "\nimplementation = CPython\nuv = 0.11.13\n"
    "version_info = 3.9.25\ninclude-system-site-packages = false\n")

os.system(B + '/python -c "import torch, pytorch_lightning as pl; '
          'print(\'READY\', torch.__version__, torch.cuda.is_available(), pl.__version__)" '
          '> /kaggle/working/ready.log 2>&1')
print()
print(open('/kaggle/working/ready.log', errors='ignore').read().strip() or "(empty - venv still broken)")

REAL INTERPRETER: /kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin/python3.9 18.6 MB
LINKED: ['activate.ps1', 'tqdm', 'transformers-cli']

READY 1.13.1+cu117 True 1.6.5


In [3]:
import os, re, time, subprocess

P = '/kaggle/working/dcpenv/bin/python'
REPO = '/kaggle/working/Deep_Correlated_Prompting'
ROOT = '/kaggle/working/dcp_results'
t0 = time.time()

subprocess.run('chmod -R +x /kaggle/working/dcpenv/bin /kaggle/working/uvpython', shell=True)

chk = ("import pyarrow as pa\n"
       "for s in ['train','dev','test']:\n"
       "    t = pa.ipc.RecordBatchFileReader(pa.memory_map('/kaggle/working/dl/arrow/hatememes_'+s+'.arrow','r')).read_all()\n"
       "    print(s, t.num_rows, t.column_names, flush=True)\n")
open('/kaggle/working/chk_hm.py', 'w').write(chk)
os.system(P + ' -u /kaggle/working/chk_hm.py > /kaggle/working/chk_hm.log 2>&1')
print("ARROW CHECK:")
print(open('/kaggle/working/chk_hm.log', errors='ignore').read().strip()[:600])

train = ('cd ' + REPO + ' && ' + P + ' -u run.py with '
         'data_root=/kaggle/working/dl/arrow log_dir=' + ROOT + ' '
         'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 task_finetune_hatememes '
         'exp_name=hm2_seed0 seed=0 max_epoch=20 '
         '> /kaggle/working/hm2_seed0.log 2>&1')
print()
print("TRAINING STARTED")
os.system(train)
print("TRAINING DONE in", round((time.time() - t0) / 60, 1), "minutes")

import glob
cks = [c for c in glob.glob(ROOT + '/hm2_seed0_seed0/version_*/checkpoints/*.ckpt') if 'last' not in c]
cks.sort(key=os.path.getmtime)
CK = cks[-1] if cks else ''
print("CHECKPOINT:", CK, round(os.path.getsize(CK) / 1e6, 1) if CK else 0, "MB")

def ev(name, extra, dev=False):
    e = ('cd ' + REPO + ' && ' + ('DCP_EVAL_ON_DEV=1 ' if dev else '') + P + ' -u run.py with '
         'data_root=/kaggle/working/dl/arrow log_dir=' + ROOT + ' '
         'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 task_finetune_hatememes '
         'exp_name=' + name + ' seed=0 test_only=True test_ratio=0.7 load_path=' + CK + ' '
         + extra + ' > /kaggle/working/' + name + '.log 2>&1')
    os.system(e)
    s = open('/kaggle/working/' + name + '.log', errors='ignore').read()
    hit = re.findall(r'DUMPED-TEST[^\n]*|the_metric[^\n]*|Traceback', s)
    print(" ", name, "->", hit[-2:] if hit else "FAILED")

if CK:
    print()
    print("TEST EVALS")
    for tt in ['both', 'image', 'text']:
        ev('hm2_test_' + tt, 'test_type=' + tt)
    print("DEV EVALS")
    for tt in ['both', 'image', 'text']:
        ev('hm2_dev_' + tt, 'test_type=' + tt, dev=True)
    print("WRONG-LABEL EVALS")
    ev('hm2_lie_image', 'test_type=image fake_missing_type=1')
    ev('hm2_lie_text', 'test_type=text fake_missing_type=2')

print()
print("DUMPS CREATED:")
for f in sorted(os.listdir('/kaggle/working/dumps')):
    if f.startswith('hm2'):
        print("  ", f, round(os.path.getsize('/kaggle/working/dumps/' + f) / 1e3, 1), "KB")
print("TOTAL MINUTES:", round((time.time() - t0) / 60, 1))

ARROW CHECK:
Traceback (most recent call last):
  File "/kaggle/working/chk_hm.py", line 3, in <module>
    t = pa.ipc.RecordBatchFileReader(pa.memory_map('/kaggle/working/dl/arrow/hatememes_'+s+'.arrow','r')).read_all()
  File "/kaggle/working/dcpenv/lib/python3.9/site-packages/pyarrow/ipc.py", line 110, in __init__
    self._open(source, footer_offset=footer_offset,
  File "pyarrow/ipc.pxi", line 942, in pyarrow.lib._RecordBatchFileReader._open
  File "pyarrow/error.pxi", line 144, in pyarrow.lib.pyarrow_internal_check_status
  File "pyarrow/error.pxi", line 100, in pyarrow.lib.check_status
pyarrow.lib.A

TRAINING STARTED
TRAINING DONE in 0.3 minutes
CHECKPOINT:  0 MB

DUMPS CREATED:
TOTAL MINUTES: 0.3


In [4]:
import os, glob, shutil, subprocess, time

t0 = time.time()
SRC = '/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data'
DST = '/kaggle/working/dl/data'

print("KAGGLE INPUTS:", os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else "none")
print("RAW HM DATASET:", os.path.exists(SRC))

if not os.path.exists(SRC):
    print()
    print("STOP - raw Hateful Memes dataset is not attached.")
    print("Add Input -> Datasets -> search 'facebook hateful meme dataset' (parthplc), attach it, then rerun this cell.")
else:
    os.makedirs(DST, exist_ok=True)
    for f in os.listdir(SRC):
        if f.endswith('.jsonl'):
            shutil.copy(SRC + '/' + f, DST + '/' + f)
    if not os.path.exists(DST + '/img'):
        os.symlink(SRC + '/img', DST + '/img')
    if not os.path.exists(DST + '/test.jsonl') or os.path.getsize(DST + '/test.jsonl') < 1000:
        os.system('wget -q -O ' + DST + '/test.jsonl '
                  'https://huggingface.co/datasets/neuralcatcher/hateful_memes/resolve/main/test_seen.jsonl')

    import json
    for split in ['train', 'dev', 'test']:
        p = DST + '/' + split + '.jsonl'
        if os.path.exists(p):
            rows = [json.loads(l) for l in open(p)]
            miss = [x for x in rows if not os.path.exists(DST + '/' + x['img'])]
            print("  ", split, len(rows), "rows | label:", 'label' in rows[0], "| missing images:", len(miss))

    print()
    print("DELETING CORRUPT ARROWS")
    for f in glob.glob('/kaggle/working/dl/arrow/hatememes_*.arrow'):
        print("  removed", os.path.basename(f), round(os.path.getsize(f)/1e6, 1), "MB")
        os.remove(f)

    os.system('pip install -q jsonlines')
    print()
    print("BUILDING ARROWS")
    os.system('cd /kaggle/working/Deep_Correlated_Prompting && python make_arrow.py '
              '--dataset hateful_memes --root /kaggle/working/dl --target /kaggle/working/dl/arrow '
              '> /kaggle/working/hm_arrow_rebuild.log 2>&1')
    print(open('/kaggle/working/hm_arrow_rebuild.log', errors='ignore').read().strip()[-800:])

    print()
    print("RESULT:")
    for f in sorted(glob.glob('/kaggle/working/dl/arrow/hatememes_*.arrow')):
        print("  ", os.path.basename(f), round(os.path.getsize(f)/1e6, 1), "MB")
    print("MINUTES:", round((time.time() - t0)/60, 1))

KAGGLE INPUTS: ['datasets']
RAW HM DATASET: False

STOP - raw Hateful Memes dataset is not attached.
Add Input -> Datasets -> search 'facebook hateful meme dataset' (parthplc), attach it, then rerun this cell.


In [1]:
import os, stat, glob

U = glob.glob('/kaggle/working/uvpython/cpython-3.9*/bin/python3.9')[0]
B = '/kaggle/working/dcpenv/bin'

for root, dirs, files in os.walk('/kaggle/working/uvpython'):
    for f in files:
        fp = os.path.join(root, f)
        if not os.path.islink(fp):
            try:
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
            except Exception:
                pass

for n in ['python', 'python3', 'python3.9']:
    t = os.path.join(B, n)
    if os.path.lexists(t):
        os.remove(t)
    os.symlink(U, t)

open('/kaggle/working/dcpenv/pyvenv.cfg', 'w').write(
    "home = " + os.path.dirname(U) + "\nimplementation = CPython\nuv = 0.11.13\n"
    "version_info = 3.9.25\ninclude-system-site-packages = false\n")

os.system(B + '/python -c "import torch, pytorch_lightning as pl; '
          'print(\'READY\', torch.__version__, torch.cuda.is_available(), pl.__version__)" '
          '> /kaggle/working/ready.log 2>&1')
print(open('/kaggle/working/ready.log', errors='ignore').read().strip() or "(venv still broken)")

print()
print("INPUT TREE:")
for root, dirs, files in os.walk('/kaggle/input'):
    d = root.replace('/kaggle/input', '')
    if d.count('/') <= 4:
        print("  ", d or '/', "|", len(files), "files |", dirs[:6])

READY 1.13.1+cu117 False 1.6.5

INPUT TREE:
   / | 0 files | ['datasets']
   /datasets | 0 files | ['johnarevalo', 'parthplc']
   /datasets/johnarevalo | 0 files | ['raw-mmimdb']
   /datasets/johnarevalo/raw-mmimdb | 0 files | ['mmimdb']
   /datasets/johnarevalo/raw-mmimdb/mmimdb | 1 files | ['mmimdb', 'dataset']
   /datasets/parthplc | 0 files | ['facebook-hateful-meme-dataset']
   /datasets/parthplc/facebook-hateful-meme-dataset | 0 files | ['data']
   /datasets/parthplc/facebook-hateful-meme-dataset/data | 5 files | ['img']


In [2]:
import os, stat, glob, shutil, json

U = glob.glob('/kaggle/working/uvpython/cpython-3.9*/bin/python3.9')[0]
B = '/kaggle/working/dcpenv/bin'

for root, dirs, files in os.walk('/kaggle/working/uvpython'):
    for f in files:
        fp = os.path.join(root, f)
        if not os.path.islink(fp):
            try:
                os.chmod(fp, os.stat(fp).st_mode | 0o111)
            except Exception:
                pass

for n in ['python', 'python3', 'python3.9']:
    t = os.path.join(B, n)
    if os.path.lexists(t):
        os.remove(t)
    os.symlink(U, t)

open('/kaggle/working/dcpenv/pyvenv.cfg', 'w').write(
    "home = " + os.path.dirname(U) + "\nimplementation = CPython\nuv = 0.11.13\n"
    "version_info = 3.9.25\ninclude-system-site-packages = false\n")

os.system(B + '/python -c "import torch; print(\'READY\', torch.__version__, torch.cuda.is_available())" > /kaggle/working/ready.log 2>&1')
print(open('/kaggle/working/ready.log', errors='ignore').read().strip() or "(venv broken)")

SRC = '/kaggle/input/datasets/parthplc/facebook-hateful-meme-dataset/data'
DST = '/kaggle/working/dl/data'
os.makedirs(DST, exist_ok=True)

for f in os.listdir(SRC):
    if f.endswith('.jsonl'):
        shutil.copy(SRC + '/' + f, DST + '/' + f)

if not os.path.exists(DST + '/img'):
    os.symlink(SRC + '/img', DST + '/img')

if not os.path.exists(DST + '/test.jsonl') or os.path.getsize(DST + '/test.jsonl') < 1000:
    os.system('wget -q -O ' + DST + '/test.jsonl https://huggingface.co/datasets/neuralcatcher/hateful_memes/resolve/main/test_seen.jsonl')

print("DATA CHECK:")
for split in ['train', 'dev', 'test']:
    p = DST + '/' + split + '.jsonl'
    if os.path.exists(p):
        rows = [json.loads(l) for l in open(p)]
        miss = [x for x in rows if not os.path.exists(DST + '/' + x['img'])]
        print("  ", split, len(rows), "rows | label:", 'label' in rows[0], "| missing:", len(miss))

READY 1.13.1+cu117 True
DATA CHECK:
   train 8500 rows | label: True | missing: 0
   dev 500 rows | label: True | missing: 0
   test 1000 rows | label: False | missing: 0


In [3]:
import os, json

DST = '/kaggle/working/dl/data'
os.remove(DST + '/test.jsonl')
os.system('wget -q -O ' + DST + '/test.jsonl '
          'https://huggingface.co/datasets/neuralcatcher/hateful_memes/resolve/main/test_seen.jsonl')

p = DST + '/test.jsonl'
print("SIZE:", os.path.getsize(p), "bytes")
rows = [json.loads(l) for l in open(p)]
miss = [x for x in rows if not os.path.exists(DST + '/' + x['img'])]
print("test", len(rows), "rows | label:", 'label' in rows[0], "| missing images:", len(miss))
print("SAMPLE:", rows[0])

SIZE: 119580 bytes
test 1000 rows | label: True | missing images: 0
SAMPLE: {'id': '16395', 'img': 'img/16395.png', 'label': 1, 'text': 'handjobs sold seperately'}


In [4]:
import os, glob, time

t0 = time.time()

print("DELETING CORRUPT ARROWS")
for f in glob.glob('/kaggle/working/dl/arrow/hatememes_*.arrow'):
    print("  removed", os.path.basename(f), round(os.path.getsize(f) / 1e6, 1), "MB")
    os.remove(f)

os.system('pip install -q jsonlines')

print()
print("BUILDING ARROWS")
os.system('cd /kaggle/working/Deep_Correlated_Prompting && python make_arrow.py '
          '--dataset hateful_memes --root /kaggle/working/dl --target /kaggle/working/dl/arrow '
          '> /kaggle/working/hm_arrow_rebuild.log 2>&1')
print(open('/kaggle/working/hm_arrow_rebuild.log', errors='ignore').read().strip()[-700:])

print()
print("RESULT:")
for f in sorted(glob.glob('/kaggle/working/dl/arrow/hatememes_*.arrow')):
    print("  ", os.path.basename(f), round(os.path.getsize(f) / 1e6, 1), "MB")

print()
print("VERIFY:")
chk = ("import pyarrow as pa\n"
       "for s in ['train','dev','test']:\n"
       "    t = pa.ipc.RecordBatchFileReader(pa.memory_map('/kaggle/working/dl/arrow/hatememes_'+s+'.arrow','r')).read_all()\n"
       "    print(s, t.num_rows, t.column_names, flush=True)\n")
open('/kaggle/working/chk_hm.py', 'w').write(chk)
os.system('/kaggle/working/dcpenv/bin/python -u /kaggle/working/chk_hm.py > /kaggle/working/chk_hm.log 2>&1')
print(open('/kaggle/working/chk_hm.log', errors='ignore').read().strip()[:600])

print("MINUTES:", round((time.time() - t0) / 60, 1))

DELETING CORRUPT ARROWS
  removed hatememes_dev.arrow 189.1 MB
  removed hatememes_train.arrow 194.5 MB
  removed hatememes_test.arrow 195.5 MB

BUILDING ARROWS
9it [00:04, 133.65it/s]
603it [00:04, 129.47it/s]
618it [00:04, 128.90it/s]
637it [00:04, 143.88it/s]
656it [00:04, 154.24it/s]
673it [00:04, 154.11it/s]
689it [00:04, 135.10it/s]
704it [00:05, 124.27it/s]
717it [00:05, 125.44it/s]
734it [00:05, 123.60it/s]
750it [00:05, 131.51it/s]
768it [00:05, 141.64it/s]
783it [00:05, 143.16it/s]
799it [00:05, 145.44it/s]
815it [00:05, 147.29it/s]
830it [00:06, 145.58it/s]
846it [00:06, 148.36it/s]
865it [00:06, 159.78it/s]
882it [00:06, 153.66it/s]
898it [00:06, 154.36it/s]
914it [00:06, 152.44it/s]
930it [00:06, 150.53it/s]
946it [00:06, 144.73it/s]
963it [00:06, 151.26it/s]
981it [00:06, 156.52it/s]
997it [00:07, 156.74it/s]
1000it [00:07, 140.63it/s]

RESULT:
   hatememes_dev.arrow 189.1 MB
   hatememes_test.arrow 361.5 MB
   hatememes_train.arrow 3054.5 MB

VERIFY:
train 8500 ['image',

In [5]:
import os, glob, shutil

def size(p):
    tot = 0
    for r, d, f in os.walk(p):
        for x in f:
            fp = os.path.join(r, x)
            if os.path.exists(fp):
                tot += os.path.getsize(fp)
    return tot / 1e9

print("BEFORE CLEANUP")
for p in ['/kaggle/working/dl/arrow', '/kaggle/working/dcp_results',
          '/kaggle/working/dumps', '/kaggle/working/dcpenv', '/kaggle/working/uvpython']:
    print("  ", p, round(size(p), 2), "GB")
print("  TOTAL /kaggle/working:", round(size('/kaggle/working'), 2), "GB")

print()
print("REMOVING FINISHED-RUN CHECKPOINTS (results already saved as dumps)")
freed = 0
for f in glob.glob('/kaggle/working/dcp_results/**/*.ckpt', recursive=True):
    freed += os.path.getsize(f)
    os.remove(f)
print("  freed", round(freed / 1e9, 2), "GB")

for d in glob.glob('/kaggle/working/dcp_results/*smoke*'):
    shutil.rmtree(d, ignore_errors=True)

print()
print("AFTER CLEANUP")
print("  TOTAL /kaggle/working:", round(size('/kaggle/working'), 2), "GB")
print("  dumps intact:", len(os.listdir('/kaggle/working/dumps')), "files")

BEFORE CLEANUP
   /kaggle/working/dl/arrow 4.48 GB
   /kaggle/working/dcp_results 4.03 GB
   /kaggle/working/dumps 0.03 GB
   /kaggle/working/dcpenv 3.85 GB
   /kaggle/working/uvpython 0.08 GB
  TOTAL /kaggle/working: 12.78 GB

REMOVING FINISHED-RUN CHECKPOINTS (results already saved as dumps)
  freed 4.03 GB

AFTER CLEANUP
  TOTAL /kaggle/working: 8.76 GB
  dumps intact: 30 files


In [6]:
import os, time, glob

P = '/kaggle/working/dcpenv/bin/python'
REPO = '/kaggle/working/Deep_Correlated_Prompting'
ROOT = '/kaggle/working/dcp_results'
t0 = time.time()

cmd = ('cd ' + REPO + ' && ' + P + ' -u run.py with '
       'data_root=/kaggle/working/dl/arrow log_dir=' + ROOT + ' '
       'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 task_finetune_hatememes '
       'exp_name=hm2_seed0 seed=0 max_epoch=20 val_check_interval=1.0 '
       '> /kaggle/working/hm2_seed0.log 2>&1')

print("TRAINING STARTED")
os.system(cmd)
print("TRAINING DONE in", round((time.time() - t0) / 60, 1), "minutes")

cks = [c for c in glob.glob(ROOT + '/hm2_seed0_seed0/version_*/checkpoints/*.ckpt') if 'last' not in c]
cks.sort(key=os.path.getmtime)
print("CHECKPOINT:", cks[-1] if cks else "NONE", round(os.path.getsize(cks[-1]) / 1e6, 1) if cks else 0, "MB")

log = open('/kaggle/working/hm2_seed0.log', errors='ignore').read()
import re
best = re.findall(r"val/the_metric[^\n]*reached[^\n]*", log)
print("BEST CHECKPOINTS DURING TRAINING:")
for b in best[-4:]:
    print("  ", b.strip()[:140])
print("COMPLETED:", 'Completed after' in log)

TRAINING STARTED
TRAINING DONE in 100.0 minutes
CHECKPOINT: /kaggle/working/dcp_results/hm2_seed0_seed0/version_0/checkpoints/epoch=7-step=272.ckpt 666.9 MB
BEST CHECKPOINTS DURING TRAINING:
   val/the_metric' reached 0.57742 (best 0.57742), saving model to '/kaggle/working/dcp_results/hm2_seed0_seed0/version_0/checkpoints/epoch=0-s
   val/the_metric' reached 0.59710 (best 0.59710), saving model to '/kaggle/working/dcp_results/hm2_seed0_seed0/version_0/checkpoints/epoch=1-s
   val/the_metric' reached 0.61555 (best 0.61555), saving model to '/kaggle/working/dcp_results/hm2_seed0_seed0/version_0/checkpoints/epoch=3-s
   val/the_metric' reached 0.63452 (best 0.63452), saving model to '/kaggle/working/dcp_results/hm2_seed0_seed0/version_0/checkpoints/epoch=7-s
COMPLETED: True


In [7]:
import os, re, time

P = '/kaggle/working/dcpenv/bin/python'
REPO = '/kaggle/working/Deep_Correlated_Prompting'
ROOT = '/kaggle/working/dcp_results'
CK = '/kaggle/working/dcp_results/hm2_seed0_seed0/version_0/checkpoints/epoch=7-step=272.ckpt'
t0 = time.time()

def ev(name, extra, dev=False):
    c = ('cd ' + REPO + ' && ' + ('DCP_EVAL_ON_DEV=1 ' if dev else '') + P + ' -u run.py with '
         'data_root=/kaggle/working/dl/arrow log_dir=' + ROOT + ' '
         'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 num_workers=4 task_finetune_hatememes '
         'exp_name=' + name + ' seed=0 test_only=True test_ratio=0.7 load_path=' + CK + ' '
         + extra + ' > /kaggle/working/' + name + '.log 2>&1')
    os.system(c)
    s = open('/kaggle/working/' + name + '.log', errors='ignore').read()
    h = re.findall(r'DUMPED-TEST[^\n]*|AUROC[^\n]*|the_metric[^\n]*|Traceback', s)
    print(" ", name, "->", h[-2:] if h else "FAILED")

print("TEST EVALS")
for tt in ['both', 'image', 'text']:
    ev('hm2_test_' + tt, 'test_type=' + tt)

print("DEV EVALS")
for tt in ['both', 'image', 'text']:
    ev('hm2_dev_' + tt, 'test_type=' + tt, dev=True)

print("WRONG-LABEL EVALS")
ev('hm2_lie_image', 'test_type=image fake_missing_type=1')
ev('hm2_lie_text', 'test_type=text fake_missing_type=2')

print()
print("DUMPS:")
for f in sorted(os.listdir('/kaggle/working/dumps')):
    if f.startswith('hm2'):
        print("  ", f, round(os.path.getsize('/kaggle/working/dumps/' + f) / 1e3, 1), "KB")
print("MINUTES:", round((time.time() - t0) / 60, 1))

TEST EVALS
  hm2_test_both -> ['AUROC_epoch       0.6431272029876709', 'the_metric            0.6431272029876709']
  hm2_test_image -> ['AUROC_epoch       0.641059398651123', 'the_metric            0.641059398651123']
  hm2_test_text -> ['AUROC_epoch       0.6925140023231506', 'the_metric            0.6925140023231506']
DEV EVALS
  hm2_dev_both -> ['AUROC_epoch       0.6345199942588806', 'the_metric            0.6345199942588806']
  hm2_dev_image -> ['AUROC_epoch       0.6240440011024475', 'the_metric            0.6240440011024475']
  hm2_dev_text -> ['AUROC_epoch       0.6383839845657349', 'the_metric            0.6383839845657349']
WRONG-LABEL EVALS
  hm2_lie_image -> ['AUROC_epoch       0.5689606070518494', 'the_metric            0.5689606070518494']
  hm2_lie_text -> ['AUROC_epoch       0.5485844612121582', 'the_metric            0.5485844612121582']

DUMPS:
   hm2_dev_both.npz 6.5 KB
   hm2_dev_image.npz 6.5 KB
   hm2_dev_text.npz 6.5 KB
   hm2_lie_image.npz 12.5 KB
   hm2_lie_tex

In [8]:
import numpy as np
from sklearn.metrics import roc_auc_score

D = '/kaggle/working/dumps/'

def load(n):
    z = np.load(D + n + '.npz')
    return z['logits'].astype(np.float64), z['labels'].astype(int)

def auroc(lg, y):
    e = np.exp(lg - lg.max(axis=1, keepdims=True))
    p = e[:, 1] / e.sum(axis=1)
    return roc_auc_score(y, p) * 100

rows = []
for tag, truth, lie in [('missing-image', 'hm2_test_image', 'hm2_lie_image'),
                        ('missing-text', 'hm2_test_text', 'hm2_lie_text')]:
    LT, yT = load(truth)
    LL, yL = load(lie)
    assert (yT == yL).all(), "label mismatch"
    diff = np.abs(LT - LL).sum(axis=1) > 1e-6
    n_aff = int(diff.sum())
    print(tag, "| samples:", len(yT), "| affected by the label:", n_aff)

    line = {}
    for acc in [1.0, 0.99, 0.95, 0.90, 0.80, 0.50, 0.0]:
        vals = []
        for rep in range(25):
            rng = np.random.RandomState(rep)
            use_truth = np.ones(len(yT), dtype=bool)
            idx = np.where(diff)[0]
            n_wrong = int(round((1 - acc) * n_aff))
            if n_wrong > 0:
                use_truth[rng.choice(idx, n_wrong, replace=False)] = False
            L = np.where(use_truth[:, None], LT, LL)
            vals.append(auroc(L, yT))
            if acc in (1.0, 0.0):
                break
        line[acc] = float(np.mean(vals))
    rows.append((tag, line))

print()
print("DETECTOR ACCURACY SENSITIVITY (AUROC)")
print("  accuracy   missing-image   missing-text")
for acc in [1.0, 0.99, 0.95, 0.90, 0.80, 0.50, 0.0]:
    print("   {:>5.0f}%      {:>8.2f}       {:>8.2f}".format(
        acc * 100, rows[0][1][acc], rows[1][1][acc]))

print()
print("RECOVERY (100% detector minus no detector)")
print("  missing-image:", round(rows[0][1][1.0] - rows[0][1][0.0], 2), "AUROC")
print("  missing-text :", round(rows[1][1][1.0] - rows[1][1][0.0], 2), "AUROC")

missing-image | samples: 1000 | affected by the label: 700
missing-text | samples: 1000 | affected by the label: 700

DETECTOR ACCURACY SENSITIVITY (AUROC)
  accuracy   missing-image   missing-text
     100%         64.11          69.25
      99%         64.02          69.07
      95%         63.78          68.20
      90%         63.28          67.42
      80%         62.28          65.96
      50%         59.83          61.68
       0%         56.90          54.86

RECOVERY (100% detector minus no detector)
  missing-image: 7.21 AUROC
  missing-text : 14.39 AUROC


In [ ]:
#ds2
import glob, os
n = 0
for f in glob.glob('/kaggle/working/dcp_results/**/last.ckpt', recursive=True):
    os.remove(f); n += 1
print("deleted", n, "last.ckpt files")
!du -sh /kaggle/working
!df -h /kaggle/working | tail -1

In [ ]:
import os, glob
print("ALL INPUTS:", os.listdir('/kaggle/input'))

base = None
for r in glob.glob('/kaggle/input/*'):
    if 'mmimdb' in os.path.basename(r).lower():
        base = r
print("\nBASE:", base)

if base:
    for d, subd, f in os.walk(base):
        depth = d[len(base):].count('/')
        print(f"{d}   dirs={subd[:5]}   files={len(f)}")
        if depth >= 2:
            subd[:] = []
else:
    print("\nMM-IMDb attach nahi hua. Add Input -> search 'raw_mmimdb' -> John Arevalo wala")

In [ ]:
import os
for d, subd, f in os.walk('/kaggle/input'):
    depth = d.count('/') - 2
    if depth <= 4:
        print(f"{d}   dirs={subd[:6]}   files={len(f)}")
    if depth >= 4:
        subd[:] = []

In [ ]:
import os
R = '/kaggle/input/datasets/johnarevalo/raw-mmimdb/mmimdb'
print("top-level files:", [x for x in os.listdir(R) if os.path.isfile(os.path.join(R, x))])

for sub in ['dataset', 'mmimdb']:
    p = os.path.join(R, sub)
    if not os.path.isdir(p):
        continue
    fs = os.listdir(p)
    ext = {}
    for x in fs:
        e = os.path.splitext(x)[1].lower()
        ext[e] = ext.get(e, 0) + 1
    print(f"\n{p}")
    print("   entries:", len(fs))
    print("   extensions:", ext)
    print("   sample:", sorted(fs)[:6])

In [ ]:
import json, os
R = '/kaggle/input/datasets/johnarevalo/raw-mmimdb/mmimdb'
D = R + '/dataset'

sp = json.load(open(R + '/split.json'))
print("split keys:", list(sp.keys()))
for k, v in sp.items():
    print(f"   {k:6s} {len(v):6d}   sample={v[:3]}")
print("   TOTAL:", sum(len(v) for v in sp.values()))

miss = 0
for k, v in sp.items():
    for s in v[:300]:
        if not (os.path.exists(f"{D}/{s}.jpeg") and os.path.exists(f"{D}/{s}.json")):
            miss += 1
print("\nmissing files among first 300 per split:", miss)

j = json.load(open(f"{D}/{sp[list(sp.keys())[0]][0]}.json"))
print("\njson fields:", list(j.keys()))
print("genres :", j.get('genres'))
print("plot[0]:", str(j.get('plot', [''])[0])[:300])

In [ ]:
%%writefile /kaggle/working/build_mmimdb_arrow.py
import json, os, io, sys
import pandas as pd
import pyarrow as pa
from PIL import Image

R = '/kaggle/input/datasets/johnarevalo/raw-mmimdb/mmimdb'
D = R + '/dataset'
OUT = '/kaggle/working/dl/arrow'
MIN_SIDE = 256
SINGLE_PLOT = True

GENRE_CLASS = ['Drama','Comedy','Romance','Thriller','Crime','Action','Adventure','Horror',
 'Documentary','Mystery','Sci-Fi','Fantasy','Family','Biography','War','History','Music',
 'Animation','Musical','Western','Sport','Short','Film-Noir']

def shrink(path):
    im = Image.open(path).convert('RGB')
    w, h = im.size
    s = min(w, h)
    if s > MIN_SIDE:
        r = MIN_SIDE / s
        im = im.resize((max(1, round(w*r)), max(1, round(h*r))), Image.BICUBIC)
    buf = io.BytesIO()
    im.save(buf, format='JPEG', quality=90)
    return buf.getvalue()

os.makedirs(OUT, exist_ok=True)
splits = json.load(open(R + '/split.json'))

for split, samples in splits.items():
    rows = []
    for n, s in enumerate(samples):
        binary = shrink(f"{D}/{s}.jpeg")
        labels = json.load(open(f"{D}/{s}.json"))
        plots = [labels['plot'][0]] if SINGLE_PLOT else labels['plot']
        genres = labels['genres']
        label = [1 if g in genres else 0 for g in GENRE_CLASS]
        rows.append((binary, plots, label, genres, s, split))
        if n % 2000 == 0:
            print(f"{split} {n}/{len(samples)}", flush=True)
    df = pd.DataFrame(rows, columns=["image","plots","label","genres","image_id","split"])
    table = pa.Table.from_pandas(df)
    with pa.OSFile(f"{OUT}/mmimdb_{split}.arrow", "wb") as sink:
        with pa.RecordBatchFileWriter(sink, table.schema) as w:
            w.write_table(table)
    print(f"WROTE {split}: {len(rows)} rows", flush=True)
    del rows, df, table

print("ALL DONE", flush=True)

In [ ]:
!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -c "print('hello'); import pyarrow, PIL, pandas; print('libs ok')"

In [ ]:
import shutil
shutil.rmtree('/kaggle/working/dcpenv', ignore_errors=True)
shutil.rmtree('/kaggle/working/uvpython', ignore_errors=True)
print("OLD ENV REMOVED")

In [ ]:
%env UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
!pip install -q uv
!uv venv --python 3.9 /kaggle/working/dcpenv
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

In [ ]:
P = '/kaggle/working/dcpenv/bin/python'
!uv pip install --python {P} -q numpy==1.23.5 setuptools==65.6.3 wheel
!uv pip install --python {P} -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117
!uv pip install --python {P} -q pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 timm==0.6.13
!uv pip install --python {P} -q sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 Pillow==9.5.0 einops ftfy jsonlines regex tqdm
print("INSTALL DONE")
!{P} -c "import torch, pytorch_lightning; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('pl', pytorch_lightning.__version__)"

In [ ]:
!/kaggle/working/dcpenv/bin/python -u /kaggle/working/build_mmimdb_arrow.py


In [ ]:
import pyarrow as pa, io
from PIL import Image

for s in ['train', 'dev', 'test']:
    t = pa.ipc.RecordBatchFileReader(
        pa.memory_map(f'/kaggle/working/dl/arrow/mmimdb_{s}.arrow', 'r')).read_all()
    print(f"{s:6s} rows={t.num_rows:6d}  cols={t.column_names}")

t = pa.ipc.RecordBatchFileReader(
    pa.memory_map('/kaggle/working/dl/arrow/mmimdb_train.arrow', 'r')).read_all()
im = Image.open(io.BytesIO(t['image'][0].as_py()))
print("\nimage size :", im.size, im.mode)
print("plots      :", len(t['plots'][0].as_py()), "->", str(t['plots'][0].as_py()[0])[:150])
print("genres     :", t['genres'][0].as_py())
print("label      :", t['label'][0].as_py())
print("label sum  :", sum(t['label'][0].as_py()))
print("image_id   :", t['image_id'][0].as_py())

lab = t['label'].to_pylist()
print("\navg genres per movie:", round(sum(sum(x) for x in lab) / len(lab), 2))
print("movies with 0 genres:", sum(1 for x in lab if sum(x) == 0))

import os
print()
!du -sh /kaggle/working/dl/arrow
!df -h /kaggle/working | tail -1

In [ ]:
import time
start = time.time()
!cd /kaggle/working/Deep_Correlated_Prompting && \
DCP_GEN=0 DCP_RETRIEVAL=0 \
/kaggle/working/dcpenv/bin/python -u run.py with \
data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 \
task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 \
max_epoch=1 seed=0 exp_name=mmimdb_smoke \
log_dir=/kaggle/working/dcp_results num_workers=4 val_check_interval=1.0 \
prompt_length=36 prompt_depth=6 both_ratio=0.5 \
"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" \
"missing_type.train=both" "missing_type.val=both" "missing_type.test=both" \
2>&1 | tee /kaggle/working/mmimdb_smoke.log
print(f"\n=== TOTAL: {(time.time()-start)/60:.1f} MINUTES ===")

In [ ]:
import pandas as pd, glob
d = sorted(glob.glob('/kaggle/working/dcp_results/mmimdb_smoke_seed0/version_*/metrics.csv'))[-1]
df = pd.read_csv(d)
print([c for c in df.columns if 'mmimdb' in c or 'metric' in c])
print(df.groupby('epoch').max()[[c for c in df.columns if 'val' in c]].to_string())

In [ ]:
import time
start = time.time()
!cd /kaggle/working/Deep_Correlated_Prompting && \
DCP_GEN=0 DCP_RETRIEVAL=0 \
/kaggle/working/dcpenv/bin/python -u run.py with \
data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 \
task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 \
max_epoch=20 seed=0 exp_name=mmimdb_base \
log_dir=/kaggle/working/dcp_results num_workers=4 val_check_interval=1.0 \
prompt_length=36 prompt_depth=6 both_ratio=0.5 \
"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" \
"missing_type.train=both" "missing_type.val=both" "missing_type.test=both" \
2>&1 | tee /kaggle/working/mmimdb_base_seed0.log
print(f"\n=== TOTAL: {(time.time()-start)/60:.1f} MINUTES ===")

In [ ]:
import glob, os, subprocess

!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -c "print('venv ok')"

print("\n--- mmimdb checkpoints ---")
for f in sorted(glob.glob('/kaggle/working/dcp_results/mmimdb_*/version_*/checkpoints/*.ckpt')):
    print(round(os.path.getsize(f)/1e6, 1), "MB", f)

print("\n--- freeing space ---")
n = 0
for f in glob.glob('/kaggle/working/dcp_results/**/last.ckpt', recursive=True):
    os.remove(f); n += 1
print("deleted", n, "last.ckpt")

import shutil
shutil.rmtree('/kaggle/working/dcp_results/mmimdb_smoke_seed0', ignore_errors=True)
print("deleted mmimdb_smoke run")

print()
!du -sh /kaggle/working
!df -h /kaggle/working | tail -1
print()
!tail -5 /kaggle/working/mmimdb_base_seed0.log

In [ ]:
import pandas as pd, glob
for d in sorted(glob.glob('/kaggle/working/dcp_results/mmimdb_base_seed0/version_*/metrics.csv')):
    df = pd.read_csv(d)
    print(d, "-> max epoch", df['epoch'].max(), "| rows", len(df))

best, bn = None, -1
for d in glob.glob('/kaggle/working/dcp_results/mmimdb_base_seed0/version_*/metrics.csv'):
    n = pd.read_csv(d)['epoch'].max()
    if n > bn:
        bn, best = n, d
print("\nUSING:", best)
df = pd.read_csv(best)
cols = ['mmimdb/val/F1_Macro_epoch', 'mmimdb/val/F1_Micro_epoch',
        'mmimdb/val/loss_epoch', 'mmimdb/train/loss']
print(df.groupby('epoch').max()[cols].to_string())

In [ ]:
import glob, os, shutil

shutil.rmtree('/kaggle/working/dcp_results/mmimdb_base_seed0/version_0', ignore_errors=True)
print("removed stale version_0")

CKPT = '/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt'
assert os.path.exists(CKPT), "checkpoint missing"
print("USING:", CKPT, flush=True)

for t in ['image', 'text', 'both']:
    cmd = (
        "cd /kaggle/working/Deep_Correlated_Prompting && "
        "DCP_GEN=0 DCP_RETRIEVAL=0 /kaggle/working/dcpenv/bin/python -u run.py with "
        "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
        "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
        "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
        f"test_only=True load_path={CKPT} exp_name=mmimdb_eval_{t} "
        '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
        '"missing_type.train=both" "missing_type.val=both" '
        f'"missing_type.test={t}" '
        f"> /kaggle/working/mmimdb_eval_{t}.log 2>&1"
    )
    os.system(cmd)
    print("DONE", t, flush=True)

print("\n===== MM-IMDb TEST RESULTS (seed 0) =====")
for t in ['image', 'text', 'both']:
    lines = open(f'/kaggle/working/mmimdb_eval_{t}.log').read().split('\n')
    hits = [l for l in lines if 'F1' in l or 'the_metric' in l]
    print(f"\n--- missing {t} ---")
    print('\n'.join(hits[-8:]) if hits else '\n'.join(lines[-12:]))

In [ ]:
%%writefile /kaggle/working/build_emb_mmimdb.py
import pyarrow as pa, io, torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

dev = 'cuda'
m = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(dev).eval()
p = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")

@torch.no_grad()
def emb(split, bs=64):
    t = pa.ipc.RecordBatchFileReader(
        pa.memory_map(f'/kaggle/working/dl/arrow/mmimdb_{split}.arrow', 'r')).read_all()
    n = t.num_rows
    I, T = [], []
    for s in range(0, n, bs):
        e = min(s + bs, n)
        imgs = [Image.open(io.BytesIO(t['image'][i].as_py())).convert('RGB') for i in range(s, e)]
        txts = [t['plots'][i].as_py()[0] for i in range(s, e)]
        pi = p(images=imgs, return_tensors='pt').to(dev)
        pt = p(text=txts, return_tensors='pt', padding=True, truncation=True, max_length=77).to(dev)
        fi = m.get_image_features(**pi)
        ft = m.get_text_features(**pt)
        I.append(torch.nn.functional.normalize(fi, dim=-1).cpu())
        T.append(torch.nn.functional.normalize(ft, dim=-1).cpu())
        if s % 3200 == 0:
            print(split, s, '/', n, flush=True)
    return {'image': torch.cat(I), 'text': torch.cat(T)}

E = {s: emb(s) for s in ['train', 'dev', 'test']}
torch.save(E, '/kaggle/working/clip_emb_mmimdb.pt')
print("SAVED", {k: (v['image'].shape, v['text'].shape) for k, v in E.items()})

In [ ]:
!/kaggle/working/dcpenv/bin/python -u /kaggle/working/build_emb_mmimdb.py

In [ ]:
import torch, pyarrow as pa, numpy as np

E = torch.load('/kaggle/working/clip_emb_mmimdb.pt')

def genres(split):
    t = pa.ipc.RecordBatchFileReader(
        pa.memory_map(f'/kaggle/working/dl/arrow/mmimdb_{split}.arrow', 'r')).read_all()
    return [set(x) for x in t['genres'].to_pylist()]

tr_g, te_g = genres('train'), genres('test')

def overlap(a_list, b_list):
    jac = [len(a & b) / len(a | b) if (a | b) else 0.0 for a, b in zip(a_list, b_list)]
    hit = [1.0 if (a & b) else 0.0 for a, b in zip(a_list, b_list)]
    return np.mean(jac), np.mean(hit) * 100

def evaluate(q, pool, name):
    idx, sims = [], []
    for s in range(0, q.shape[0], 512):
        S = q[s:s+512] @ pool.T
        v, i = S.max(dim=1)
        idx.append(i); sims.append(v)
    idx = torch.cat(idx).tolist()
    sims = torch.cat(sims)
    rj, rh = overlap(te_g, [tr_g[i] for i in idx])
    g = torch.Generator().manual_seed(0)
    rnd = torch.randint(0, len(tr_g), (len(te_g),), generator=g).tolist()
    bj, bh = overlap(te_g, [tr_g[i] for i in rnd])
    print(f"\n=== {name} ===")
    print(f"  mean Jaccard       retrieved {rj:.3f}    random {bj:.3f}")
    print(f"  shares >=1 genre   retrieved {rh:.1f}%    random {bh:.1f}%")
    print(f"  mean cosine of match: {sims.mean():.3f}")

evaluate(E['test']['text'], E['train']['image'], "text query -> train image   (MISSING IMAGE case)")
evaluate(E['test']['image'], E['train']['text'], "image query -> train text   (MISSING TEXT case)")

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/datasets/mmimdb_dataset.py'
print(open(p).read())

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/datasets/hatememes_dataset.py'
s = open(p).read()
i = s.index('import')
j = s.index('def __getitem__')
print("=========== IMPORTS + __init__ ===========")
print(s[i:j])
print("=========== __getitem__ ===========")
print(s[j:])

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/datasets/mmimdb_dataset.py'
src = open(p).read()

if 'DCP_RETRIEVAL' in src:
    print("ALREADY PATCHED")
else:
    old_imp = "import torch\nimport random\nimport os"
    assert old_imp in src, "import block not found"
    src = src.replace(old_imp, "import torch\nimport random, os, io\nimport pyarrow as pa\nfrom PIL import Image", 1)

    a = src.index("        self.missing_table = missing_table")
    b = src.index("    def __getitem__", a)
    init_block = '''        self.missing_table = missing_table

        self.retrieval = os.environ.get('DCP_RETRIEVAL', '0') == '1'
        self.retrieval_k = int(os.environ.get('DCP_RETRIEVAL_K', '1'))
        if self.retrieval:
            E = torch.load('/kaggle/working/clip_emb_mmimdb.pt')
            key = {'train': 'train', 'val': 'dev', 'test': 'test'}[split]
            self.q_img = E[key]['image']
            self.q_txt = E[key]['text']
            self.pool_img = E['train']['image']
            self.pool_txt = E['train']['text']
            self.pool_table = pa.ipc.RecordBatchFileReader(
                pa.memory_map('/kaggle/working/dl/arrow/mmimdb_train.arrow', 'r')).read_all()
            self.is_train_split = (split == 'train')
            print(f">>> MMIMDB RETRIEVAL IS ON | split={split} | k={self.retrieval_k}", flush=True)

    def _topk(self, query, pool, own_index):
        sims = pool @ query
        if self.is_train_split:
            sims[own_index] = -1e9
        return torch.topk(sims, self.retrieval_k).indices.tolist()

'''
    src = src[:a] + init_block + src[b:]

    a = src.index("        # missing image, dummy image is all-one image")
    b = src.index('        labels = self.table["label"][image_index].as_py()', a)
    get_block = '''        # missing image
        if self.missing_table[image_index] == 2 or simulate_missing_type == 2:
            if self.retrieval:
                js = self._topk(self.q_txt[image_index], self.pool_img, image_index)
                stack = []
                for j in js:
                    rimg = Image.open(io.BytesIO(self.pool_table['image'][j].as_py())).convert('RGB')
                    stack.append([tr(rimg) for tr in self.transforms])
                image_tensor = [torch.stack([s[k] for s in stack]).mean(0)
                                for k in range(len(stack[0]))]
            else:
                for idx in range(len(image_tensor)):
                    image_tensor[idx] = torch.ones(image_tensor[idx].size()).float()

        # missing text
        if self.missing_table[image_index] == 1 or simulate_missing_type == 1:
            if self.retrieval:
                js = self._topk(self.q_img[image_index], self.pool_txt, image_index)
                text = ' '.join(self.pool_table['plots'][j][0].as_py() for j in js)
            else:
                text = ''
        else:
            text = self.get_text(index)["text"]

'''
    src = src[:a] + get_block + src[b:]
    open(p, 'w').write(src)
    print("PATCH APPLIED")

m = open('/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py').read()
print("infer() _retr guard present:", "_retr" in m)
!/kaggle/working/dcpenv/bin/python -m py_compile {p} && echo "SYNTAX OK"
print()
print(open(p).read()[-1800:])

In [ ]:
import shutil
shutil.rmtree('/kaggle/working/dcpenv', ignore_errors=True)
shutil.rmtree('/kaggle/working/uvpython', ignore_errors=True)
print("OLD ENV REMOVED")

In [ ]:
%env UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
!pip install -q uv
!uv venv --python 3.9 /kaggle/working/dcpenv
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

In [ ]:
P = '/kaggle/working/dcpenv/bin/python'
!uv pip install --python {P} -q numpy==1.23.5 setuptools==65.6.3 wheel
!uv pip install --python {P} -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117
!uv pip install --python {P} -q pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 timm==0.6.13
!uv pip install --python {P} -q sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 Pillow==9.5.0 einops ftfy jsonlines regex tqdm
print("INSTALL DONE")
!{P} -c "import torch, pytorch_lightning; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('pl', pytorch_lightning.__version__)"
!{P} -m py_compile /kaggle/working/Deep_Correlated_Prompting/clip/datasets/mmimdb_dataset.py && echo "SYNTAX OK"

In [ ]:
import time
start = time.time()
!cd /kaggle/working/Deep_Correlated_Prompting && \
DCP_RETRIEVAL=1 DCP_RETRIEVAL_K=1 DCP_GEN=0 \
/kaggle/working/dcpenv/bin/python -u run.py with \
data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 \
task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 \
max_epoch=20 seed=0 exp_name=mmimdb_retr \
log_dir=/kaggle/working/dcp_results num_workers=4 val_check_interval=1.0 \
prompt_length=36 prompt_depth=6 both_ratio=0.5 \
"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" \
"missing_type.train=both" "missing_type.val=both" "missing_type.test=both" \
2>&1 | tee /kaggle/working/mmimdb_retr_seed0.log
print(f"\n=== TOTAL: {(time.time()-start)/60:.1f} MINUTES ===")

In [ ]:
import glob, os

!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
ok = os.popen('/kaggle/working/dcpenv/bin/python -c "print(1)"').read().strip()
print("venv:", "OK" if ok == "1" else "BROKEN")

print("\n--- log tail ---")
!tail -4 /kaggle/working/mmimdb_retr_seed0.log

ck = [f for f in glob.glob('/kaggle/working/dcp_results/mmimdb_retr_seed0/version_*/checkpoints/*.ckpt')
      if 'last' not in os.path.basename(f)]
print("\ncheckpoints:", ck)

if ok == "1" and ck:
    CKPT = sorted(ck)[-1]
    print("\nUSING:", CKPT, flush=True)
    for t in ['image', 'text', 'both']:
        cmd = (
            "cd /kaggle/working/Deep_Correlated_Prompting && "
            "DCP_RETRIEVAL=1 DCP_RETRIEVAL_K=1 DCP_GEN=0 "
            "/kaggle/working/dcpenv/bin/python -u run.py with "
            "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
            "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
            "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
            f"test_only=True load_path={CKPT} exp_name=mmimdb_retr_eval_{t} "
            '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
            '"missing_type.train=both" "missing_type.val=both" '
            f'"missing_type.test={t}" '
            f"> /kaggle/working/mmimdb_retr_eval_{t}.log 2>&1"
        )
        os.system(cmd)
        print("DONE", t, flush=True)

    print("\n===== MM-IMDb RETRIEVAL TEST RESULTS =====")
    for t in ['image', 'text', 'both']:
        lines = open(f'/kaggle/working/mmimdb_retr_eval_{t}.log').read().split('\n')
        hits = [l for l in lines if 'F1' in l or 'the_metric' in l]
        print(f"\n--- missing {t} ---")
        print('\n'.join(hits[-6:]) if hits else '\n'.join(lines[-10:]))
else:
    print("\nEvaluation skipped - venv ya checkpoint missing")

In [ ]:
o = open('/kaggle/working/Deep_Correlated_Prompting/clip/modules/objectives.py').read()
i = o.index('def compute_mmimdb')
j = o.index('\ndef ', i + 10)
print(o[i:j])
print("=" * 60)
m = open('/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py').read()
k = m.index('def validation_epoch_end')
print(m[k:k+400])

In [ ]:
B = '/kaggle/working/Deep_Correlated_Prompting/clip/modules'

o_path = f'{B}/objectives.py'
o = open(o_path).read()
if '_dump_logits' in o:
    print("objectives already patched")
else:
    dump = '''    if pl_module.hparams.config["test_only"]:
        if not hasattr(pl_module, "_dump_logits"):
            pl_module._dump_logits, pl_module._dump_labels = [], []
        pl_module._dump_logits.append(imgcls_logits.detach().float().cpu())
        pl_module._dump_labels.append(imgcls_labels.detach().float().cpu())

'''
    for task in ['hatememes', 'mmimdb']:
        anchor = f'    loss = getattr(pl_module, f"{{phase}}_{task}_loss")'
        assert anchor in o, f"anchor missing for {task}"
        o = o.replace(anchor, dump + anchor, 1)
    open(o_path, 'w').write(o)
    print("OBJECTIVES PATCHED")

m_path = f'{B}/clip_missing_aware_prompt_module.py'
m = open(m_path).read()
if '_dump_logits' in m:
    print("module already patched")
else:
    a = m.index('    def validation_epoch_end(self, outs):')
    b = m.index('clip_utils.epoch_wrapup(self)', a) + len('clip_utils.epoch_wrapup(self)')
    ins = '''
        if self.hparams.config["test_only"] and getattr(self, "_dump_logits", None):
            import numpy as np
            os.makedirs('/kaggle/working/dumps', exist_ok=True)
            nm = self.hparams.config["exp_name"]
            np.savez('/kaggle/working/dumps/' + nm + '.npz',
                     logits=torch.cat(self._dump_logits).numpy(),
                     labels=torch.cat(self._dump_labels).numpy())
            print(">>> DUMPED " + nm, flush=True)
            self._dump_logits, self._dump_labels = [], []'''
    m = m[:b] + ins + m[b:]
    open(m_path, 'w').write(m)
    print("MODULE PATCHED")

!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -m py_compile {o_path} {m_path} && echo "SYNTAX OK"

In [ ]:
import os

BASE = '/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt'
RETR = '/kaggle/working/dcp_results/mmimdb_retr_seed0/version_0/checkpoints/epoch=11-step=732.ckpt'

for tag, ckpt, retr in [('base', BASE, '0'), ('retr', RETR, '1')]:
    for t in ['image', 'text', 'both']:
        name = f'mmimdb_{tag}_{t}'
        cmd = (
            "cd /kaggle/working/Deep_Correlated_Prompting && "
            f"DCP_RETRIEVAL={retr} DCP_RETRIEVAL_K=1 DCP_GEN=0 "
            "/kaggle/working/dcpenv/bin/python -u run.py with "
            "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
            "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
            "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
            f"test_only=True load_path={ckpt} exp_name={name} "
            '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
            '"missing_type.train=both" "missing_type.val=both" '
            f'"missing_type.test={t}" '
            f"> /kaggle/working/dump_{name}.log 2>&1"
        )
        os.system(cmd)
        print("DONE", name, flush=True)

import glob
print("\ndumps:")
for f in sorted(glob.glob('/kaggle/working/dumps/*.npz')):
    print("  ", f, round(os.path.getsize(f)/1e6, 1), "MB")

In [ ]:
m = open('/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py').read()
k = m.index('def test_step')
print(m[k:k+900])
print("=" * 60)
!tail -6 /kaggle/working/dump_mmimdb_base_image.log

In [ ]:
m_path = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
m = open(m_path).read()

if 'DUMPED-TEST' in m:
    print("test_epoch_end already patched")
else:
    a = m.index('    def test_epoch_end(self, outs):')
    b = m.index('clip_utils.epoch_wrapup(self)', a) + len('clip_utils.epoch_wrapup(self)')
    ins = '''
        if getattr(self, "_dump_logits", None):
            import numpy as np
            os.makedirs('/kaggle/working/dumps', exist_ok=True)
            nm = self.hparams.config["exp_name"]
            np.savez('/kaggle/working/dumps/' + nm + '.npz',
                     logits=torch.cat(self._dump_logits).numpy(),
                     labels=torch.cat(self._dump_labels).numpy())
            print(">>> DUMPED-TEST " + nm, flush=True)
            self._dump_logits, self._dump_labels = [], []'''
    m = m[:b] + ins + m[b:]
    open(m_path, 'w').write(m)
    print("TEST_EPOCH_END PATCHED")

!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
ok = os.system(f'/kaggle/working/dcpenv/bin/python -m py_compile {m_path}')
print("SYNTAX OK" if ok == 0 else "SYNTAX ERROR")

In [ ]:
import os

m_path = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
m = open(m_path).read()
print("file length:", len(m))
print("has test_epoch_end   :", 'def test_epoch_end' in m)
print("has dump accumulator :", '_dump_logits' in m)
print("has DUMPED-TEST      :", 'DUMPED-TEST' in m)

if 'DUMPED-TEST' not in m:
    a = m.index('    def test_epoch_end(self, outs):')
    b = m.index('clip_utils.epoch_wrapup(self)', a) + len('clip_utils.epoch_wrapup(self)')
    ins = '''
        if getattr(self, "_dump_logits", None):
            import numpy as np
            os.makedirs('/kaggle/working/dumps', exist_ok=True)
            nm = self.hparams.config["exp_name"]
            np.savez('/kaggle/working/dumps/' + nm + '.npz',
                     logits=torch.cat(self._dump_logits).numpy(),
                     labels=torch.cat(self._dump_labels).numpy())
            print(">>> DUMPED-TEST " + nm, flush=True)
            self._dump_logits, self._dump_labels = [], []'''
    m = m[:b] + ins + m[b:]
    open(m_path, 'w').write(m)
    print("PATCH WRITTEN")

print("\nverify after write:", 'DUMPED-TEST' in open(m_path).read())

In [ ]:
!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -c "print('venv alive')"
!/kaggle/working/dcpenv/bin/python -m py_compile /kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py && echo "SYNTAX OK"

In [ ]:
import os, glob

BASE = '/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt'
RETR = '/kaggle/working/dcp_results/mmimdb_retr_seed0/version_0/checkpoints/epoch=11-step=732.ckpt'

for tag, ckpt, retr in [('base', BASE, '0'), ('retr', RETR, '1')]:
    for t in ['image', 'text', 'both']:
        name = f'mmimdb_{tag}_{t}'
        cmd = (
            "cd /kaggle/working/Deep_Correlated_Prompting && "
            f"DCP_RETRIEVAL={retr} DCP_RETRIEVAL_K=1 DCP_GEN=0 "
            "/kaggle/working/dcpenv/bin/python -u run.py with "
            "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
            "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
            "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
            f"test_only=True load_path={ckpt} exp_name={name} "
            '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
            '"missing_type.train=both" "missing_type.val=both" '
            f'"missing_type.test={t}" '
            f"> /kaggle/working/dump_{name}.log 2>&1"
        )
        os.system(cmd)
        print("DONE", name, flush=True)

print("\ndumps:")
d = sorted(glob.glob('/kaggle/working/dumps/*.npz'))
for f in d:
    print("  ", os.path.basename(f), round(os.path.getsize(f)/1e6, 1), "MB")
if not d:
    print("  (none) -- last log tail:")
    !tail -20 /kaggle/working/dump_mmimdb_base_image.log

In [ ]:
import os, glob
!df -h /kaggle/working | tail -1
!du -sh /kaggle/working/* 2>/dev/null | sort -h | tail -12
print("\nlog files:")
for f in sorted(glob.glob('/kaggle/working/dump_*.log')):
    print("  ", os.path.basename(f), os.path.getsize(f), "bytes")
print("\ndumps dir exists:", os.path.exists('/kaggle/working/dumps'))

In [ ]:
import glob, os, shutil
n = 0
for f in glob.glob('/kaggle/working/dcp_results/**/last.ckpt', recursive=True):
    os.remove(f); n += 1
print("deleted", n, "last.ckpt")

for d in glob.glob('/kaggle/working/dcp_results/*eval*') + glob.glob('/kaggle/working/dcp_results/*lie*'):
    shutil.rmtree(d, ignore_errors=True)
print("removed eval run folders")

!df -h /kaggle/working | tail -1

In [ ]:
!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!cd /kaggle/working/Deep_Correlated_Prompting && \
DCP_RETRIEVAL=0 DCP_GEN=0 \
/kaggle/working/dcpenv/bin/python -u run.py with \
data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 \
task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 \
prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 \
test_only=True exp_name=mmimdb_base_image \
load_path=/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt \
"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" \
"missing_type.train=both" "missing_type.val=both" "missing_type.test=image" \
2>&1 | tail -30

In [ ]:
print("python works")
!echo "shell works"
!ls /kaggle/working | head -5
!/kaggle/working/dcpenv/bin/python -c "print('venv works')"

In [1]:
import shutil
shutil.rmtree('/kaggle/working/dcpenv', ignore_errors=True)
shutil.rmtree('/kaggle/working/uvpython', ignore_errors=True)
print("OLD ENV REMOVED")

OLD ENV REMOVED


In [2]:
%env UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
!pip install -q uv
!uv venv --python 3.9 /kaggle/working/dcpenv
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

env: UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 50.2 MB/s eta 0:00:00:00:0100:01
Using CPython 3.9.25
Creating virtual environment at: dcpenv
Activate with: source dcpenv/bin/activate
hello


In [3]:
P = '/kaggle/working/dcpenv/bin/python'
!uv pip install --python {P} -q numpy==1.23.5 setuptools==65.6.3 wheel
!uv pip install --python {P} -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117
!uv pip install --python {P} -q pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 timm==0.6.13
!uv pip install --python {P} -q sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 Pillow==9.5.0 einops ftfy jsonlines regex tqdm
print("INSTALL DONE")
!{P} -c "import torch, pytorch_lightning; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('pl', pytorch_lightning.__version__)"

INSTALL DONE
torch 1.13.1+cu117 cuda False
pl 1.6.5


In [ ]:
import os, glob

BASE = '/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt'
RETR = '/kaggle/working/dcp_results/mmimdb_retr_seed0/version_0/checkpoints/epoch=11-step=732.ckpt'

for tag, ckpt, retr in [('base', BASE, '0'), ('retr', RETR, '1')]:
    for t in ['image', 'text', 'both']:
        name = f'mmimdb_{tag}_{t}'
        cmd = (
            "cd /kaggle/working/Deep_Correlated_Prompting && "
            f"DCP_RETRIEVAL={retr} DCP_RETRIEVAL_K=1 DCP_GEN=0 "
            "/kaggle/working/dcpenv/bin/python -u run.py with "
            "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
            "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
            "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
            f"test_only=True load_path={ckpt} exp_name={name} "
            '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
            '"missing_type.train=both" "missing_type.val=both" '
            f'"missing_type.test={t}" '
            f"> /kaggle/working/dump_{name}.log 2>&1"
        )
        os.system(cmd)
        print("DONE", name, flush=True)

print("\ndumps:")
d = sorted(glob.glob('/kaggle/working/dumps/*.npz'))
for f in d:
    print("  ", os.path.basename(f), round(os.path.getsize(f)/1e6, 1), "MB")
if not d:
    print("  (none)")
    !tail -20 /kaggle/working/dump_mmimdb_base_image.log

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             average_precision_score, hamming_loss,
                             accuracy_score, roc_auc_score)

GEN = ['Drama','Comedy','Romance','Thriller','Crime','Action','Adventure','Horror',
       'Documentary','Mystery','Sci-Fi','Fantasy','Family','Biography','War','History',
       'Music','Animation','Musical','Western','Sport','Short','Film-Noir']

def sig(x): return 1 / (1 + np.exp(-x))

rows, per_genre, support = [], {}, None
for tag in ['base', 'retr']:
    for t in ['image', 'text', 'both']:
        d = np.load(f'/kaggle/working/dumps/mmimdb_{tag}_{t}.npz')
        L, Y = d['logits'], d['labels'].astype(int)
        P = sig(L); B = (P > 0.5).astype(int)
        support = Y.sum(0)
        try:
            auroc = roc_auc_score(Y, P, average='macro') * 100
        except Exception:
            auroc = float('nan')
        rows.append({
            'model': tag, 'missing': t,
            'F1_macro':    f1_score(Y, B, average='macro',    zero_division=0) * 100,
            'F1_micro':    f1_score(Y, B, average='micro',    zero_division=0) * 100,
            'F1_weighted': f1_score(Y, B, average='weighted', zero_division=0) * 100,
            'F1_samples':  f1_score(Y, B, average='samples',  zero_division=0) * 100,
            'Prec_macro':  precision_score(Y, B, average='macro', zero_division=0) * 100,
            'Rec_macro':   recall_score(Y, B, average='macro',    zero_division=0) * 100,
            'mAP':         average_precision_score(Y, P, average='macro') * 100,
            'AUROC_macro': auroc,
            'Hamming':     hamming_loss(Y, B) * 100,
            'SubsetAcc':   accuracy_score(Y, B) * 100,
        })
        per_genre[f'{tag}_{t}'] = f1_score(Y, B, average=None, zero_division=0) * 100

pd.set_option('display.width', 250)
df = pd.DataFrame(rows).set_index(['model', 'missing']).round(2)
print("===== MM-IMDb : FULL METRIC SUITE (test, seed 0, eta=70%) =====\n")
print(df.to_string())

pg = pd.DataFrame(per_genre, index=GEN).round(1)
pg.insert(0, 'support', support)
pg = pg.sort_values('support', ascending=False)
print("\n\n===== PER-GENRE F1 (23 genres) =====\n")
print(pg.to_string())

df.to_csv('/kaggle/working/mmimdb_metrics.csv')
pg.to_csv('/kaggle/working/mmimdb_per_genre_f1.csv')
print("\nsaved: mmimdb_metrics.csv, mmimdb_per_genre_f1.csv")

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

def sig(x): return 1 / (1 + np.exp(-x))

out = []
for tag in ['base', 'retr']:
    for t in ['image', 'text', 'both']:
        d = np.load(f'/kaggle/working/dumps/mmimdb_{tag}_{t}.npz')
        P, Y = sig(d['logits']), d['labels'].astype(int)

        f_05 = f1_score(Y, (P > 0.5).astype(int), average='macro', zero_division=0) * 100

        grid = np.arange(0.05, 0.71, 0.01)
        f_glob = max(f1_score(Y, (P > g).astype(int), average='macro', zero_division=0) * 100
                     for g in grid)

        thr = np.full(P.shape[1], 0.5)
        for c in range(P.shape[1]):
            best, bt = -1, 0.5
            for g in grid:
                s = f1_score(Y[:, c], (P[:, c] > g).astype(int), zero_division=0)
                if s > best:
                    best, bt = s, g
            thr[c] = bt
        f_per = f1_score(Y, (P > thr).astype(int), average='macro', zero_division=0) * 100

        out.append({'model': tag, 'missing': t, 'F1@0.5': f_05,
                    'F1@best_global': f_glob, 'F1@per_class': f_per,
                    'gain_global': f_glob - f_05, 'gain_perclass': f_per - f_05})

print(pd.DataFrame(out).set_index(['model', 'missing']).round(2).to_string())

In [ ]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/datasets/mmimdb_dataset.py'
s = open(p).read()
if 'DCP_EVAL_ON_DEV' in s:
    print("already patched")
else:
    old = '''        elif split == "test":
            names = ["mmimdb_test"]'''
    assert old in s, "anchor not found"
    new = '''        elif split == "test":
            if os.environ.get('DCP_EVAL_ON_DEV', '0') == '1':
                names = ["mmimdb_dev"]
                print(">>> EVALUATING ON DEV SPLIT", flush=True)
            else:
                names = ["mmimdb_test"]'''
    s = s.replace(old, new, 1)
    open(p, 'w').write(s)
    print("PATCHED")
!/kaggle/working/dcpenv/bin/python -m py_compile {p} && echo "SYNTAX OK"

In [ ]:
import os, glob
BASE = '/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt'
RETR = '/kaggle/working/dcp_results/mmimdb_retr_seed0/version_0/checkpoints/epoch=11-step=732.ckpt'

for tag, ckpt, retr in [('base', BASE, '0'), ('retr', RETR, '1')]:
    for t in ['image', 'text', 'both']:
        name = f'devmmimdb_{tag}_{t}'
        cmd = (
            "cd /kaggle/working/Deep_Correlated_Prompting && "
            f"DCP_EVAL_ON_DEV=1 DCP_RETRIEVAL={retr} DCP_RETRIEVAL_K=1 DCP_GEN=0 "
            "/kaggle/working/dcpenv/bin/python -u run.py with "
            "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
            "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
            "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
            f"test_only=True load_path={ckpt} exp_name={name} "
            '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
            '"missing_type.train=both" "missing_type.val=both" '
            f'"missing_type.test={t}" '
            f"> /kaggle/working/dump_{name}.log 2>&1"
        )
        os.system(cmd)
        print("DONE", name, flush=True)

print("\ndev dumps:")
for f in sorted(glob.glob('/kaggle/working/dumps/devmmimdb_*.npz')):
    print("  ", os.path.basename(f), round(os.path.getsize(f)/1e6, 1), "MB")

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             hamming_loss, accuracy_score)

def sig(x): return 1 / (1 + np.exp(-x))
grid = np.arange(0.05, 0.71, 0.01)

rows = []
for tag in ['base', 'retr']:
    for t in ['image', 'text', 'both']:
        dv = np.load(f'/kaggle/working/dumps/devmmimdb_{tag}_{t}.npz')
        te = np.load(f'/kaggle/working/dumps/mmimdb_{tag}_{t}.npz')
        Pd, Yd = sig(dv['logits']), dv['labels'].astype(int)
        Pt, Yt = sig(te['logits']), te['labels'].astype(int)

        g_best = max(grid, key=lambda g: f1_score(Yd, (Pd > g).astype(int),
                                                  average='macro', zero_division=0))
        thr = np.full(Pd.shape[1], 0.5)
        for c in range(Pd.shape[1]):
            thr[c] = max(grid, key=lambda g: f1_score(Yd[:, c], (Pd[:, c] > g).astype(int),
                                                      zero_division=0))

        B05, Bg, Bp = (Pt > 0.5).astype(int), (Pt > g_best).astype(int), (Pt > thr).astype(int)
        f = lambda B, a='macro': f1_score(Yt, B, average=a, zero_division=0) * 100

        rows.append({
            'model': tag, 'missing': t,
            'F1@0.5': f(B05),
            'F1 dev-global': f(Bg), 'thr_g': round(g_best, 2),
            'F1 dev-perclass': f(Bp),
            'gain_global': f(Bg) - f(B05),
            'gain_perclass': f(Bp) - f(B05),
            'Prec_pc': precision_score(Yt, Bp, average='macro', zero_division=0) * 100,
            'Rec_pc': recall_score(Yt, Bp, average='macro', zero_division=0) * 100,
            'F1micro_pc': f(Bp, 'micro'),
            'Hamming_pc': hamming_loss(Yt, Bp) * 100,
            'SubsetAcc_pc': accuracy_score(Yt, Bp) * 100,
        })

pd.set_option('display.width', 250)
df = pd.DataFrame(rows).set_index(['model', 'missing']).round(2)
print("===== MM-IMDb : threshold tuned on DEV, applied to TEST =====\n")
print(df.to_string())
df.to_csv('/kaggle/working/mmimdb_threshold_tuned.csv')
print("\nsaved: mmimdb_threshold_tuned.csv")

In [4]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
src = open(p).read()

if 'gen_gate_img' in src:
    print("ALREADY PATCHED")
else:
    anchor = "self.mmimdb_classifier.apply(objectives.init_weights)"
    assert anchor in src
    ins = """

        if os.environ.get('DCP_GATE', '0') == '1':
            self.gen_gate_img = nn.Parameter(torch.zeros(hidden_size // 2))
            self.gen_gate_txt = nn.Parameter(torch.zeros(hidden_size // 2))
            print(">>> GATES CREATED | init 0 = exact baseline", flush=True)
"""
    k = src.index(anchor) + len(anchor)
    src = src[:k] + ins + src[k:]

    old = "        _gen = os.environ.get('DCP_GEN', '0') == '1'"
    assert old in src
    src = src.replace(old, old + "\n        _gate = os.environ.get('DCP_GATE', '0') == '1'", 1)

    marker = "        else:\n            for idx in range(len(img)):"
    assert marker in src
    gate_branch = """        elif _gate:
            mt = torch.tensor([int(v) for v in batch["missing_type"]], device=both_feats.device)
            img_half = both_feats[:, :feature_dim]
            txt_half = both_feats[:, feature_dim:]
            m1 = (mt == 1).unsqueeze(1).to(both_feats.dtype)
            m2 = (mt == 2).unsqueeze(1).to(both_feats.dtype)
            gi = self.gen_gate_img.to(both_feats.dtype).unsqueeze(0)
            gt = self.gen_gate_txt.to(both_feats.dtype).unsqueeze(0)
            new_img = img_half * (1 - m2) + (img_half * gi) * m2
            new_txt = txt_half * (1 - m1) + (txt_half * gt) * m1
            both_feats = torch.cat([new_img, new_txt], dim=1)
"""
    src = src.replace(marker, gate_branch + marker, 1)
    open(p, 'w').write(src)
    print("PATCH APPLIED")

!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -m py_compile {p} && echo "SYNTAX OK"

PATCH APPLIED
SYNTAX OK


In [5]:
import time
start = time.time()
!cd /kaggle/working/Deep_Correlated_Prompting && \
DCP_RETRIEVAL=1 DCP_RETRIEVAL_K=1 DCP_GATE=1 DCP_GEN=0 DCP_GEN_LR_MULT=0.1 \
/kaggle/working/dcpenv/bin/python -u run.py with \
data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 \
task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 \
max_epoch=20 seed=0 exp_name=mmimdb_gate \
log_dir=/kaggle/working/dcp_results num_workers=4 val_check_interval=1.0 \
prompt_length=36 prompt_depth=6 both_ratio=0.5 \
"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" \
"missing_type.train=both" "missing_type.val=both" "missing_type.test=both" \
2>&1 | tee /kaggle/working/mmimdb_gate.log
print(f"\n=== TOTAL: {(time.time()-start)/60:.1f} MINUTES ===")

WARNING - root - Changed type of config entry "max_steps" from int to NoneType
WARNING - CLIP - No observers have been added to this run
INFO - CLIP - Running command 'main'
INFO - CLIP - Started
Global seed set to 0
INFO - torch.distributed.nn.jit.instantiator - Created a temporary directory at /tmp/tmpx_rgf_tq
INFO - torch.distributed.nn.jit.instantiator - Writing /tmp/tmpx_rgf_tq/_remote_module_non_scriptable.py
100%|████████████████████████████████████████| 351M/351M [00:03<00:00, 104MiB/s]
Building custom CLIP
>>> GATES CREATED | init 0 = exact baseline
/kaggle/working/dcpenv/lib/python3.9/site-packages/torchmetrics/utilities/prints.py:36: UserWarning: Torchmetrics v0.9 introduced a new argument class property called `full_state_update` that has
                not been set for this class (F1_Score). The property determines if `update` by
                default needs access to the full metric state. If this is not the case, significant speedups can be
                achieved and

In [1]:
import numpy as np, torch, pyarrow as pa
from sklearn.metrics import f1_score

E = torch.load('/kaggle/working/clip_emb_mmimdb.pt')

def labels_of(split):
    t = pa.ipc.RecordBatchFileReader(
        pa.memory_map(f'/kaggle/working/dl/arrow/mmimdb_{split}.arrow', 'r')).read_all()
    return np.array(t['label'].to_pylist(), dtype=np.float32)

TR = labels_of('train')
def sig(x): return 1 / (1 + np.exp(-x))

def knn_prior(q, pool, K):
    q = q.numpy(); pool = pool.numpy()
    out = np.zeros((len(q), TR.shape[1]), dtype=np.float32)
    for s in range(0, len(q), 512):
        S = q[s:s+512] @ pool.T
        idx = np.argpartition(-S, K, axis=1)[:, :K]
        out[s:s+512] = TR[idx].mean(1)
    return out
QUERY = {'image': 'text', 'text': 'image'}

for t in ['image', 'text']:
    mod = QUERY[t]
    dv = np.load(f'/kaggle/working/dumps/devmmimdb_base_{t}.npz')
    te = np.load(f'/kaggle/working/dumps/mmimdb_base_{t}.npz')
    Pd, Yd = sig(dv['logits']), dv['labels'].astype(int)
    Pt, Yt = sig(te['logits']), te['labels'].astype(int)

    best = (-1, None, None)
    for K in [1, 5, 10, 20, 50]:
        Kd = knn_prior(E['dev'][mod],  E['train'][mod], K)
        for a in np.arange(0.5, 1.01, 0.05):
            s = f1_score(Yd, ((a*Pd + (1-a)*Kd) > 0.5).astype(int),
                         average='macro', zero_division=0)
            if s > best[0]:
                best = (s, K, round(float(a), 2))

    _, K, a = best
    Kt = knn_prior(E['test'][mod], E['train'][mod], K)
    base = f1_score(Yt, (Pt > 0.5).astype(int), average='macro', zero_division=0) * 100
    blend = f1_score(Yt, ((a*Pt + (1-a)*Kt) > 0.5).astype(int), average='macro', zero_division=0) * 100
    print(f"missing-{t:5s} | best K={K}, alpha={a} | baseline {base:.2f} -> kNN-blend {blend:.2f} "
          f"| gain {blend-base:+.2f}")

missing-image | best K=1, alpha=0.95 | baseline 53.73 -> kNN-blend 53.77 | gain +0.04
missing-text  | best K=1, alpha=1.0 | baseline 49.69 -> kNN-blend 49.69 | gain +0.00


In [2]:
!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -c "print('venv ok')"

In [3]:
import os

CKPT = '/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt'
runs = [('image', None), ('image', 1), ('text', None), ('text', 2)]

for t, fmt in runs:
    tag = f"{t}_{fmt}"
    extra = f"fake_missing_type={fmt} " if fmt is not None else ""
    cmd = (
        "cd /kaggle/working/Deep_Correlated_Prompting && "
        "DCP_RETRIEVAL=0 DCP_GEN=0 DCP_GATE=0 /kaggle/working/dcpenv/bin/python -u run.py with "
        "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
        "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
        "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
        f"test_only=True load_path={CKPT} exp_name=mmimdb_lie_{tag} {extra}"
        '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
        '"missing_type.train=both" "missing_type.val=both" '
        f'"missing_type.test={t}" '
        f"> /kaggle/working/mmimdb_lie_{tag}.log 2>&1"
    )
    os.system(cmd)
    print("DONE", tag, flush=True)

print("\n===== MM-IMDb GAP 2 : wrong missing type =====")
for t, fmt in runs:
    tag = f"{t}_{fmt}"
    lines = open(f'/kaggle/working/mmimdb_lie_{tag}.log').read().split('\n')
    hits = [l for l in lines if 'F1_Macro' in l or 'F1_Micro' in l]
    label = "truth" if fmt is None else f"LIE (told {fmt})"
    print(f"\n--- actually missing {t} | {label} ---")
    print('\n'.join(hits[-2:]))

DONE image_None
DONE image_1
DONE text_None
DONE text_2

===== MM-IMDb GAP 2 : wrong missing type =====

--- actually missing image | truth ---


--- actually missing image | LIE (told 1) ---


--- actually missing text | truth ---


--- actually missing text | LIE (told 2) ---



In [4]:
import os, glob
!/kaggle/working/dcpenv/bin/python -c "print('venv ok')"
print("\nlog sizes:")
for f in sorted(glob.glob('/kaggle/working/mmimdb_lie_*.log')):
    print("  ", os.path.basename(f), os.path.getsize(f), "bytes")
print("\n--- tail of one log ---")
!tail -25 /kaggle/working/mmimdb_lie_image_None.log


log sizes:
   mmimdb_lie_image_1.log 0 bytes
   mmimdb_lie_image_None.log 0 bytes
   mmimdb_lie_text_2.log 0 bytes
   mmimdb_lie_text_None.log 0 bytes

--- tail of one log ---


In [5]:
import shutil
shutil.rmtree('/kaggle/working/dcpenv', ignore_errors=True)
shutil.rmtree('/kaggle/working/uvpython', ignore_errors=True)
print("OLD ENV REMOVED")

OLD ENV REMOVED


In [6]:
%env UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
!pip install -q uv
!uv venv --python 3.9 /kaggle/working/dcpenv
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

env: UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 77.1 MB/s eta 0:00:00:00:0100:01
Using CPython 3.9.25
Creating virtual environment at: dcpenv
Activate with: source dcpenv/bin/activate
hello


In [7]:
P = '/kaggle/working/dcpenv/bin/python'
!uv pip install --python {P} -q numpy==1.23.5 setuptools==65.6.3 wheel
!uv pip install --python {P} -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117
!uv pip install --python {P} -q pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 timm==0.6.13
!uv pip install --python {P} -q sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 Pillow==9.5.0 einops ftfy jsonlines regex tqdm
print("INSTALL DONE")
!{P} -c "import torch, pytorch_lightning; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('pl', pytorch_lightning.__version__)"

INSTALL DONE
torch 1.13.1+cu117 cuda True
pl 1.6.5


In [8]:
import os

CKPT = '/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt'
runs = [('image', None), ('image', 1), ('text', None), ('text', 2)]

for t, fmt in runs:
    tag = f"{t}_{fmt}"
    extra = f"fake_missing_type={fmt} " if fmt is not None else ""
    cmd = (
        "cd /kaggle/working/Deep_Correlated_Prompting && "
        "DCP_RETRIEVAL=0 DCP_GEN=0 DCP_GATE=0 /kaggle/working/dcpenv/bin/python -u run.py with "
        "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
        "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
        "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
        f"test_only=True load_path={CKPT} exp_name=mmimdb_lie_{tag} {extra}"
        '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
        '"missing_type.train=both" "missing_type.val=both" '
        f'"missing_type.test={t}" '
        f"> /kaggle/working/mmimdb_lie_{tag}.log 2>&1"
    )
    os.system(cmd)
    print("DONE", tag, flush=True)

print("\n===== MM-IMDb GAP 2 : wrong missing type =====")
for t, fmt in runs:
    tag = f"{t}_{fmt}"
    lines = open(f'/kaggle/working/mmimdb_lie_{tag}.log').read().split('\n')
    hits = [l for l in lines if 'F1_Macro' in l or 'F1_Micro' in l]
    label = "truth" if fmt is None else f"LIE (told {fmt})"
    print(f"\n--- actually missing {t} | {label} ---")
    print('\n'.join(hits[-2:]) if hits else "(no output - check log)")

DONE image_None
DONE image_1
DONE text_None
DONE text_2

===== MM-IMDb GAP 2 : wrong missing type =====

--- actually missing image | truth ---
 mmimdb/val/F1_Macro_epoch       0.5372720956802368
 mmimdb/val/F1_Micro_epoch       0.6145550608634949

--- actually missing image | LIE (told 1) ---
 mmimdb/val/F1_Macro_epoch      0.27903684973716736
 mmimdb/val/F1_Micro_epoch       0.4363780915737152

--- actually missing text | truth ---
 mmimdb/val/F1_Macro_epoch       0.4968940019607544
 mmimdb/val/F1_Micro_epoch       0.6009376645088196

--- actually missing text | LIE (told 2) ---
 mmimdb/val/F1_Macro_epoch      0.25782305002212524
 mmimdb/val/F1_Micro_epoch       0.2348189502954483


In [7]:
import shutil
shutil.rmtree('/kaggle/working/dcpenv', ignore_errors=True)
shutil.rmtree('/kaggle/working/uvpython', ignore_errors=True)
print("OLD ENV REMOVED")

OLD ENV REMOVED


In [8]:
%env UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
!pip install -q uv
!uv venv --python 3.9 /kaggle/working/dcpenv
!/kaggle/working/dcpenv/bin/python -c "print('hello')"

env: UV_PYTHON_INSTALL_DIR=/kaggle/working/uvpython
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 69.8 MB/s eta 0:00:00:00:0100:01
Using CPython 3.9.25
Creating virtual environment at: dcpenv
Activate with: source dcpenv/bin/activate
hello


In [9]:
P = '/kaggle/working/dcpenv/bin/python'
!uv pip install --python {P} -q numpy==1.23.5 setuptools==65.6.3 wheel
!uv pip install --python {P} -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --index-url https://download.pytorch.org/whl/cu117
!uv pip install --python {P} -q pytorch_lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 timm==0.6.13
!uv pip install --python {P} -q sacred==0.8.2 pyarrow==12.0.1 pandas==1.5.3 Pillow==9.5.0 einops ftfy jsonlines regex tqdm
print("INSTALL DONE")
!{P} -c "import torch, pytorch_lightning; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('pl', pytorch_lightning.__version__)"
!{P} -m py_compile /kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py && echo "SYNTAX OK"

INSTALL DONE
torch 1.13.1+cu117 cuda True
pl 1.6.5
SYNTAX OK


In [11]:
p = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
src = open(p).read()

if 'DCP_DETECT' in src:
    print("ALREADY PATCHED")
else:
    anchor = '        both_feats = self.model(img, text, batch["missing_type"])'
    assert anchor in src, "anchor not found"
    det = '''        if os.environ.get('DCP_DETECT', '0') == '1':
            _pred = []
            for _k in range(len(img)):
                _ch = img[_k].reshape(img[_k].shape[0], -1)
                _im = bool(_ch.std(dim=1).max().item() < 1e-3)
                if torch.is_tensor(text):
                    _tm = bool(int((text[_k] != 0).sum().item()) <= 3)
                else:
                    _tm = (len(str(text[_k]).strip()) == 0)
                _pred.append(2 if _im else (1 if _tm else 0))
            _true = [int(v) for v in batch["missing_type"]]
            if not hasattr(self, '_det_ok'):
                self._det_ok, self._det_n = 0, 0
            self._det_ok += sum(int(a == b) for a, b in zip(_pred, _true))
            self._det_n += len(_pred)
            batch["missing_type"] = _pred

'''
    src = src.replace(anchor, det + anchor, 1)

    a = src.index('    def test_epoch_end(self, outs):')
    b = src.index('clip_utils.epoch_wrapup(self)', a) + len('clip_utils.epoch_wrapup(self)')
    acc = '''
        if getattr(self, "_det_n", 0) > 0:
            print(">>> DETECTOR ACCURACY " + str(round(100.0 * self._det_ok / self._det_n, 2))
                  + "%  (" + str(self._det_ok) + "/" + str(self._det_n) + ")", flush=True)
            self._det_ok, self._det_n = 0, 0'''
    src = src[:b] + acc + src[b:]
    open(p, 'w').write(src)
    print("PATCH APPLIED")

!chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin
!/kaggle/working/dcpenv/bin/python -m py_compile {p} && echo "SYNTAX OK"

ALREADY PATCHED
SYNTAX OK


In [12]:
import os

CKPT = '/kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt'

for t in ['image', 'text', 'both']:
    name = f'mmimdb_detect_{t}'
    cmd = (
        "cd /kaggle/working/Deep_Correlated_Prompting && "
        "DCP_DETECT=1 DCP_RETRIEVAL=0 DCP_GEN=0 DCP_GATE=0 "
        "/kaggle/working/dcpenv/bin/python -u run.py with "
        "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
        "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
        "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
        f"test_only=True load_path={CKPT} exp_name={name} "
        '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
        '"missing_type.train=both" "missing_type.val=both" '
        f'"missing_type.test={t}" '
        f"> /kaggle/working/{name}.log 2>&1"
    )
    os.system(cmd)
    print("DONE", t, flush=True)

print("\n===== MM-IMDb : DETECTOR (no oracle label given) =====")
for t in ['image', 'text', 'both']:
    lines = open(f'/kaggle/working/mmimdb_detect_{t}.log').read().split('\n')
    hits = [l for l in lines if 'F1_Macro' in l or 'DETECTOR ACCURACY' in l]
    print(f"\n--- missing {t} ---")
    print('\n'.join(hits[-3:]) if hits else "(check log)")

DONE image
DONE text
DONE both

===== MM-IMDb : DETECTOR (no oracle label given) =====

--- missing image ---
(check log)

--- missing text ---
(check log)

--- missing both ---
(check log)


In [13]:
s = open('/kaggle/working/mmimdb_detect_image.log').read()
print("LOG LENGTH:", len(s))
print(s[-3000:])

LOG LENGTH: 7780
 to the full metric state. If this is not the case, significant speedups can be
                achieved and we recommend setting this to `False`.
                We provide an checking function
                `from torchmetrics.utilities import check_forward_full_state_property`
                that can be used to check if the `full_state_update=True` (old and potential slower behaviour,
                default for now) or if `full_state_update=False` can be used safely.
                
  warnings.warn(*args, **kwargs)
ERROR - CLIP - Failed after 0:00:07!
Traceback (most recent call last):
  File "/kaggle/working/dcpenv/lib/python3.9/site-packages/sacred/experiment.py", line 312, in run_commandline
    return self.run(
  File "/kaggle/working/dcpenv/lib/python3.9/site-packages/sacred/experiment.py", line 276, in run
    run()
  File "/kaggle/working/dcpenv/lib/python3.9/site-packages/sacred/run.py", line 238, in __call__
    self.result = self.main_function(*args)
 

In [14]:
import glob, os, torch

for f in sorted(glob.glob('/kaggle/working/dcp_results/**/*.ckpt', recursive=True)):
    mb = os.path.getsize(f) / 1e6
    try:
        torch.load(f, map_location='cpu')
        ok = "OK"
    except Exception as e:
        ok = "CORRUPT (" + type(e).__name__ + ")"
    print(f"{mb:8.1f} MB  {ok:22s}  {f}")
    

   194.6 MB  CORRUPT (RuntimeError)  /kaggle/working/dcp_results/base_fixed_seed0/version_0/checkpoints/epoch=7-step=272.ckpt
   194.3 MB  CORRUPT (RuntimeError)  /kaggle/working/dcp_results/base_seed1_seed1/version_0/checkpoints/epoch=8-step=306.ckpt
   194.0 MB  CORRUPT (RuntimeError)  /kaggle/working/dcp_results/gen2_seed0_seed0/version_0/checkpoints/epoch=1-step=68.ckpt
   195.9 MB  CORRUPT (RuntimeError)  /kaggle/working/dcp_results/gen2_smoke_seed0/version_0/checkpoints/epoch=0-step=34.ckpt
   195.2 MB  CORRUPT (RuntimeError)  /kaggle/working/dcp_results/gen_seed0_seed0/version_0/checkpoints/epoch=0-step=34.ckpt
   193.6 MB  CORRUPT (RuntimeError)  /kaggle/working/dcp_results/gen_seed0_seed0/version_6/checkpoints/epoch=0-step=34.ckpt
   194.1 MB  CORRUPT (RuntimeError)  /kaggle/working/dcp_results/mmimdb_base_seed0/version_1/checkpoints/epoch=8-step=549.ckpt
   194.5 MB  CORRUPT (RuntimeError)  /kaggle/working/dcp_results/mmimdb_retr_seed0/version_0/checkpoints/epoch=11-step=732.

In [16]:
s = open('/kaggle/working/mmimdb_base_v2.log').read()
print("LEN:", len(s))
print(s[-2500:])

LEN: 8268
setup_hook
    self.datamodule.setup(stage=fn)
  File "/kaggle/working/Deep_Correlated_Prompting/clip/datamodules/multitask_datamodule.py", line 33, in setup
    dm.setup(stage)
  File "/kaggle/working/Deep_Correlated_Prompting/clip/datamodules/mmimdb_datamodule.py", line 19, in setup
    super().setup(stage)
  File "/kaggle/working/Deep_Correlated_Prompting/clip/datamodules/datamodule_base.py", line 119, in setup
    self.set_train_dataset()
  File "/kaggle/working/Deep_Correlated_Prompting/clip/datamodules/datamodule_base.py", line 60, in set_train_dataset
    self.train_dataset = self.dataset_cls(
  File "/kaggle/working/Deep_Correlated_Prompting/clip/datasets/mmimdb_dataset.py", line 23, in __init__
    super().__init__(
  File "/kaggle/working/Deep_Correlated_Prompting/clip/datasets/base_dataset.py", line 45, in __init__
    tables = [
  File "/kaggle/working/Deep_Correlated_Prompting/clip/datasets/base_dataset.py", line 46, in <listcomp>
    pa.ipc.RecordBatchFileReader

In [17]:
import os, glob
import pyarrow as pa

print("=== ARROW ===")
for f in sorted(glob.glob('/kaggle/working/dl/arrow/*.arrow')):
    mb = os.path.getsize(f)/1e6
    try:
        t = pa.ipc.RecordBatchFileReader(pa.memory_map(f, 'r')).read_all()
        s = f"OK  rows={t.num_rows}"
    except Exception as e:
        s = "CORRUPT"
    print(f"{mb:9.1f} MB  {s:20s}  {os.path.basename(f)}")

print("\n=== EMBEDDINGS ===")
import torch
for f in ['/kaggle/working/clip_emb_mmimdb.pt', '/kaggle/working/clip_emb.pt']:
    if os.path.exists(f):
        mb = os.path.getsize(f)/1e6
        try:
            E = torch.load(f); s = "OK  " + str(list(E.keys()))
        except Exception:
            s = "CORRUPT"
        print(f"{mb:9.1f} MB  {s}  {os.path.basename(f)}")

print("\n=== DUMPS (logits) ===")
import numpy as np
for f in sorted(glob.glob('/kaggle/working/dumps/*.npz')):
    try:
        d = np.load(f); s = f"OK  {d['logits'].shape}"
    except Exception:
        s = "CORRUPT"
    print(f"  {s:22s}  {os.path.basename(f)}")

print()
!df -h /kaggle/working | tail -1

=== ARROW ===
    189.1 MB  OK  rows=500          hatememes_dev.arrow
    195.5 MB  CORRUPT               hatememes_test.arrow
    194.5 MB  CORRUPT               hatememes_train.arrow
     86.9 MB  OK  rows=2608         mmimdb_dev.arrow
    194.9 MB  CORRUPT               mmimdb_test.arrow
    195.0 MB  CORRUPT               mmimdb_train.arrow

=== EMBEDDINGS ===
    106.3 MB  OK  ['train', 'dev', 'test']  clip_emb_mmimdb.pt
     41.0 MB  OK  ['train', 'dev', 'test']  clip_emb.pt

=== DUMPS (logits) ===
  OK  (2608, 23)          devmmimdb_base_both.npz
  OK  (2608, 23)          devmmimdb_base_image.npz
  OK  (2608, 23)          devmmimdb_base_text.npz
  OK  (2608, 23)          devmmimdb_retr_both.npz
  OK  (2608, 23)          devmmimdb_retr_image.npz
  OK  (2608, 23)          devmmimdb_retr_text.npz
  OK  (7799, 23)          mmimdb_base_both.npz
  OK  (7799, 23)          mmimdb_base_image.npz
  OK  (7799, 23)          mmimdb_base_text.npz
  OK  (7799, 23)          mmimdb_lie_image_1.n

In [18]:
import numpy as np, torch, glob, os
from sklearn.metrics import f1_score

def sig(x): return 1 / (1 + np.exp(-x))
MTDIR = '/kaggle/working/Deep_Correlated_Prompting/datasets/missing_tables'
print("missing tables:", [os.path.basename(f) for f in glob.glob(MTDIR + '/mmimdb_test*')])

for scen, fake in [('image', 1), ('text', 2)]:
    o = np.load(f'/kaggle/working/dumps/mmimdb_lie_{scen}_None.npz')
    l = np.load(f'/kaggle/working/dumps/mmimdb_lie_{scen}_{fake}.npz')
    Y  = o['labels'].astype(int)
    Po, Pl = sig(o['logits']), sig(l['logits'])

    mt = torch.load(f'{MTDIR}/mmimdb_test_missing_{scen}_07.pt').numpy()
    nz = mt != 0
    print(f"\n=== missing-{scen} | {int(nz.sum())} of {len(mt)} samples actually missing ===")
    print(f"{'detector acc':>13s} {'F1-Macro':>10s} {'vs wrong label':>15s}")

    f1_wrong = None
    for p in [1.00, 0.99, 0.95, 0.90, 0.80, 0.50, 0.00]:
        rng = np.random.default_rng(0)
        wrong = nz & (rng.random(len(Y)) > p)
        P = np.where(wrong[:, None], Pl, Po)
        f1 = f1_score(Y, (P > 0.5).astype(int), average='macro', zero_division=0) * 100
        if p == 0.00:
            f1_wrong = f1
        print(f"{p*100:12.0f}% {f1:10.2f} {'' if f1_wrong is None else f'{f1-f1_wrong:+14.2f}'}")

missing tables: ['mmimdb_test_missing_image_07.pt', 'mmimdb_test_missing_text_07.pt', 'mmimdb_test_missing_both_07.pt']

=== missing-image | 5459 of 7799 samples actually missing ===
 detector acc   F1-Macro  vs wrong label
         100%      53.73 
          99%      53.61 
          95%      52.87 
          90%      51.95 
          80%      49.74 
          50%      42.96 
           0%      27.90          +0.00

=== missing-text | 5459 of 7799 samples actually missing ===
 detector acc   F1-Macro  vs wrong label
         100%      49.69 
          99%      49.39 
          95%      48.57 
          90%      47.64 
          80%      45.66 
          50%      39.71 
           0%      25.78          +0.00


In [19]:
import os, glob, shutil, time, re
import pandas as pd

os.system('chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin')
for f in glob.glob('/kaggle/working/dcp_results/**/*.ckpt', recursive=True):
    os.remove(f)
print("cleaned", flush=True)

PY = '/kaggle/working/dcpenv/bin/python'
CD = "cd /kaggle/working/Deep_Correlated_Prompting && "

# 1) arrow dobara
t0 = time.time()
os.system(f"{PY} -u /kaggle/working/build_mmimdb_arrow.py > /kaggle/working/rebuild_arrow.log 2>&1")
print(f"ARROW REBUILT in {(time.time()-t0)/60:.1f} min", flush=True)

COMMON = (
    "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
    "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 "
    "prompt_length=36 prompt_depth=6 both_ratio=0.5 num_workers=4 "
    '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
    '"missing_type.train=both" "missing_type.val=both" '
)

# 2) seed 1 training
t0 = time.time()
os.system(CD + f"DCP_RETRIEVAL=0 DCP_GEN=0 DCP_GATE=0 DCP_DETECT=0 {PY} -u run.py with " +
          COMMON + '"missing_type.test=both" max_epoch=20 seed=1 exp_name=mmimdb_s1 '
          "log_dir=/kaggle/working/dcp_results val_check_interval=1.0 "
          "> /kaggle/working/mmimdb_s1.log 2>&1")
print(f"TRAINING DONE in {(time.time()-t0)/60:.1f} min", flush=True)

ck = [f for f in glob.glob('/kaggle/working/dcp_results/mmimdb_s1_seed1/version_*/checkpoints/*.ckpt')
      if 'last' not in os.path.basename(f)]
assert ck, "no checkpoint produced - check /kaggle/working/mmimdb_s1.log"
CKPT = sorted(ck)[-1]
print("CKPT:", CKPT, flush=True)

# 3) saare evals
jobs = [(f's1_oracle_{t}', t, "", "DCP_DETECT=0") for t in ['image', 'text', 'both']]
jobs += [('s1_lie_image', 'image', "fake_missing_type=1 ", "DCP_DETECT=0"),
         ('s1_lie_text',  'text',  "fake_missing_type=2 ", "DCP_DETECT=0")]
jobs += [(f's1_detect_{t}', t, "", "DCP_DETECT=1") for t in ['image', 'text', 'both']]

for name, t, extra, env in jobs:
    os.system(CD + f"{env} DCP_RETRIEVAL=0 DCP_GEN=0 DCP_GATE=0 {PY} -u run.py with " +
              COMMON + f'"missing_type.test={t}" test_only=True load_path={CKPT} '
              f"exp_name={name} {extra}> /kaggle/working/{name}.log 2>&1")
    print("DONE", name, flush=True)

def grab(name, key='F1_Macro'):
    try:
        s = open(f'/kaggle/working/{name}.log').read()
        m = re.findall(rf'mmimdb/val/{key}_epoch\s+([\d.]+)', s)
        return round(float(m[-1]) * 100, 2) if m else None
    except Exception:
        return None

def det_acc(name):
    try:
        m = re.findall(r'DETECTOR ACCURACY ([\d.]+)%', open(f'/kaggle/working/{name}.log').read())
        return m[-1] if m else None
    except Exception:
        return None

rows = []
for t in ['image', 'text', 'both']:
    rows.append({
        'scenario': t,
        'seed0_oracle': {'image': 53.73, 'text': 49.69, 'both': 51.44}[t],
        'seed1_oracle': grab(f's1_oracle_{t}'),
        'seed1_lie': grab('s1_lie_image') if t == 'image' else (grab('s1_lie_text') if t == 'text' else None),
        'seed1_detector': grab(f's1_detect_{t}'),
        'detector_acc': det_acc(f's1_detect_{t}'),
    })
df = pd.DataFrame(rows)
df['seed_spread'] = (df['seed1_oracle'] - df['seed0_oracle']).abs()
df['recovery'] = df['seed1_detector'] - df['seed1_lie']
print("\n" + "=" * 80)
print(df.to_string(index=False))
print("=" * 80)
df.to_csv('/kaggle/working/gap3_final_results.csv', index=False)
print("saved: /kaggle/working/gap3_final_results.csv")

cleaned
ARROW REBUILT in 22.2 min
TRAINING DONE in 191.3 min
CKPT: /kaggle/working/dcp_results/mmimdb_s1_seed1/version_0/checkpoints/epoch=7-step=488.ckpt
DONE s1_oracle_image
DONE s1_oracle_text
DONE s1_oracle_both
DONE s1_lie_image
DONE s1_lie_text
DONE s1_detect_image
DONE s1_detect_text
DONE s1_detect_both

scenario  seed0_oracle  seed1_oracle  seed1_lie seed1_detector detector_acc  seed_spread recovery
   image         53.73         52.87      27.62           None         None         0.86      NaN
    text         49.69         48.97      25.99           None         None         0.72      NaN
    both         51.44         51.10        NaN           None         None         0.34      NaN
saved: /kaggle/working/gap3_final_results.csv


In [1]:
import os
p = '/kaggle/working/Deep_Correlated_Prompting/clip/modules/clip_missing_aware_prompt_module.py'
src = open(p).read()

if 'gen_mgate' in src:
    print("ALREADY PATCHED")
else:
    anchor = "self.mmimdb_classifier.apply(objectives.init_weights)"
    assert anchor in src
    ins = """

        if os.environ.get('DCP_MGATE', '0') == '1':
            self.gen_mgate = nn.Linear(hidden_size, hidden_size // 2)
            nn.init.zeros_(self.gen_mgate.weight)
            nn.init.zeros_(self.gen_mgate.bias)
            print(">>> MODALITY GATE CREATED | init g=0.5 = exact baseline", flush=True)
"""
    k = src.index(anchor) + len(anchor)
    src = src[:k] + ins + src[k:]

    old = "        _gate = os.environ.get('DCP_GATE', '0') == '1'"
    assert old in src
    src = src.replace(old, old + "\n        _mgate = os.environ.get('DCP_MGATE', '0') == '1'", 1)

    marker = "        else:\n            for idx in range(len(img)):"
    assert marker in src
    branch = """        elif _mgate:
            mt = torch.tensor([int(v) for v in batch["missing_type"]], device=both_feats.device)
            g = torch.sigmoid(self.gen_mgate(both_feats.float())).to(both_feats.dtype)
            img_half = both_feats[:, :feature_dim] * (2.0 * g)
            txt_half = both_feats[:, feature_dim:] * (2.0 * (1.0 - g))
            m1 = (mt == 1).unsqueeze(1).to(both_feats.dtype)
            m2 = (mt == 2).unsqueeze(1).to(both_feats.dtype)
            img_half = img_half * (1 - m2)
            txt_half = txt_half * (1 - m1)
            both_feats = torch.cat([img_half, txt_half], dim=1)
            if self.training and not hasattr(self, '_mg_dbg'):
                print(">>> MGATE mean g=" + str(round(float(g.float().mean()), 4)), flush=True)
                self._mg_dbg = True
"""
    src = src.replace(marker, branch + marker, 1)
    open(p, 'w').write(src)
    print("PATCH APPLIED")

os.system('chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin')
assert os.system(f'/kaggle/working/dcpenv/bin/python -m py_compile {p}') == 0
print("SYNTAX OK", flush=True)

os.system("cd /kaggle/working/Deep_Correlated_Prompting && "
          "DCP_MGATE=1 DCP_RETRIEVAL=0 DCP_GEN=0 DCP_GATE=0 DCP_DETECT=0 DCP_GEN_LR_MULT=0.1 "
          "/kaggle/working/dcpenv/bin/python -u run.py with "
          "data_root=/kaggle/working/dl/arrow num_gpus=1 num_nodes=1 "
          "task_finetune_mmimdb per_gpu_batchsize=4 batch_size=256 max_epoch=1 seed=0 "
          "exp_name=mg_smoke log_dir=/kaggle/working/dcp_results num_workers=4 "
          "val_check_interval=1.0 prompt_length=36 prompt_depth=6 both_ratio=0.5 "
          '"missing_ratio.train=0.7" "missing_ratio.val=0.7" "missing_ratio.test=0.7" '
          '"missing_type.train=both" "missing_type.val=both" "missing_type.test=both" '
          "> /kaggle/working/mg_smoke.log 2>&1")

import re
s = open('/kaggle/working/mg_smoke.log').read()
for pat in [r'>>> MODALITY GATE CREATED.*', r'>>> GEN PARAM GROUP.*', r'>>> MGATE mean g=.*',
            r'Trainable params.*', r'val/the_metric.*']:
    m = re.findall(pat, s)
    print(m[-1] if m else f"MISSING: {pat}")
if 'Error' in s or 'Traceback' in s:
    print(s[-1500:])

PATCH APPLIED
SYNTAX OK
MISSING: >>> MODALITY GATE CREATED.*
MISSING: >>> GEN PARAM GROUP.*
MISSING: >>> MGATE mean g=.*
MISSING: Trainable params.*
MISSING: val/the_metric.*


In [1]:
import os

p = '/kaggle/working/mg_smoke.log'
if os.path.exists(p):
    s = open(p, errors='ignore').read()
    print("LOG LENGTH:", len(s))
    print("LOG TAIL:")
    print(s[-3000:] if s else "(empty file)")
else:
    print("LOG FILE DOES NOT EXIST")

os.system('chmod -R +x /kaggle/working/uvpython /kaggle/working/dcpenv/bin 2>/dev/null')
print("PYTHON PROBE:", os.popen('/kaggle/working/dcpenv/bin/python -c "import torch,pytorch_lightning; print(torch.__version__, torch.cuda.is_available(), pytorch_lightning.__version__)" 2>&1').read().strip())

ck = '/kaggle/working/dcp_results/mmimdb_s1_seed1/version_0/checkpoints/epoch=7-step=488.ckpt'
print("CKPT EXISTS:", os.path.exists(ck), "SIZE MB:", round(os.path.getsize(ck)/1e6, 1) if os.path.exists(ck) else 0)

for f in ['mmimdb_train.arrow', 'mmimdb_dev.arrow', 'mmimdb_test.arrow']:
    fp = '/kaggle/working/datasets/' + f
    print(f, os.path.exists(fp), round(os.path.getsize(fp)/1e6, 1) if os.path.exists(fp) else 0)

LOG LENGTH: 0
LOG TAIL:
(empty file)
PYTHON PROBE: 
CKPT EXISTS: True SIZE MB: 667.2
mmimdb_train.arrow False 0
mmimdb_dev.arrow False 0
mmimdb_test.arrow False 0


In [2]:
import os

print("=== ARROW FILES ANYWHERE ===")
found = 0
for root, dirs, files in os.walk('/kaggle'):
    if '/dcpenv/' in root or '/uvpython/' in root:
        dirs[:] = []
        continue
    for f in files:
        if f.endswith('.arrow'):
            fp = os.path.join(root, f)
            print(fp, round(os.path.getsize(fp)/1e6, 1), "MB")
            found += 1
print("TOTAL ARROW FILES:", found)

print()
print("=== /kaggle/working TOP LEVEL ===")
for f in sorted(os.listdir('/kaggle/working')):
    fp = '/kaggle/working/' + f
    if os.path.isdir(fp):
        print("[DIR ]", f)
    else:
        print("[FILE]", f, round(os.path.getsize(fp)/1e6, 2), "MB")

print()
print("=== VENV STATE ===")
print("dcpenv exists:", os.path.exists('/kaggle/working/dcpenv'))
print("uvpython exists:", os.path.exists('/kaggle/working/uvpython'))
print("uv binary:", os.popen('which uv 2>&1').read().strip())

print()
print("=== SAVED SETUP SCRIPTS ===")
for f in sorted(os.listdir('/kaggle/working')):
    if f.endswith('.py') or f.endswith('.sh') or f.endswith('.txt'):
        print("---", f, "---")
        print(open('/kaggle/working/' + f, errors='ignore').read()[:600])

=== ARROW FILES ANYWHERE ===
/kaggle/working/dl/arrow/hatememes_test.arrow 195.5 MB
/kaggle/working/dl/arrow/mmimdb_dev.arrow 86.9 MB
/kaggle/working/dl/arrow/hatememes_train.arrow 194.5 MB
/kaggle/working/dl/arrow/hatememes_dev.arrow 189.1 MB
/kaggle/working/dl/arrow/mmimdb_test.arrow 263.3 MB
/kaggle/working/dl/arrow/mmimdb_train.arrow 524.8 MB
TOTAL ARROW FILES: 6

=== /kaggle/working TOP LEVEL ===
[DIR ] .virtual_documents
[DIR ] Deep_Correlated_Prompting
[FILE] build_emb.py 0.0 MB
[FILE] build_emb_mmimdb.py 0.0 MB
[FILE] build_mmimdb_arrow.py 0.0 MB
[FILE] clip_emb.pt 40.96 MB
[FILE] clip_emb_mmimdb.pt 106.33 MB
[DIR ] dcp_results
[DIR ] dcpenv
[DIR ] dl
[FILE] dump_devhm_base_both.log 0.0 MB
[FILE] dump_devhm_base_image.log 0.0 MB
[FILE] dump_devhm_base_text.log 0.0 MB
[FILE] dump_devhm_retr_both.log 0.0 MB
[FILE] dump_devhm_retr_image.log 0.0 MB
[FILE] dump_devhm_retr_text.log 0.0 MB
[FILE] dump_devmmimdb_base_both.log 0.09 MB
[FILE] dump_devmmimdb_base_image.log 0.09 MB
[FILE] 

In [3]:
import os, glob, shutil, subprocess

SP = glob.glob('/kaggle/working/dcpenv/lib/python*/site-packages')
ver = {}
if SP:
    for d in os.listdir(SP[0]):
        if d.endswith('.dist-info') and '-' in d[:-10]:
            name, v = d[:-10].rsplit('-', 1)
            ver[name.lower().replace('_', '-')] = v
print("RECOVERED PACKAGE RECORDS:", len(ver))

KEY = ['torch','torchvision','pytorch-lightning','torchmetrics','transformers','sacred',
       'pyarrow','timm','einops','ftfy','regex','numpy','pillow','scikit-learn','scipy',
       'pandas','tqdm','sentencepiece','tokenizers','jsonlines','opencv-python-headless']
for k in KEY:
    print("  ", k, "=", ver.get(k, "not found"))

r = subprocess.run('echo SHELL_OK', shell=True, capture_output=True, text=True)
print("SHELL TEST:", r.stdout.strip() or "SHELL IS DEAD")

shutil.rmtree('/kaggle/working/dcpenv', ignore_errors=True)
env = dict(os.environ, UV_PYTHON_INSTALL_DIR='/kaggle/working/uvpython')

def run(c, tag):
    r = subprocess.run(c, shell=True, capture_output=True, text=True, env=env)
    print("STEP", tag, "-> exit", r.returncode)
    tail = (r.stdout[-600:] + r.stderr[-600:]).strip()
    if tail:
        print(tail)

run('uv python install 3.9', 'PYTHON39')
run('uv venv --python 3.9 /kaggle/working/dcpenv', 'VENV')

P = '/kaggle/working/dcpenv/bin/python'
spec = ' '.join(k + '==' + ver[k] for k in KEY if k in ver)
print("INSTALL SPEC:", spec)
run('uv pip install --python ' + P + ' ' + spec, 'INSTALL')
run('chmod -R +x /kaggle/working/dcpenv/bin /kaggle/working/uvpython', 'CHMOD')
run(P + ' -c "import torch, pytorch_lightning, transformers, sacred, pyarrow; print(\'VENV OK\', torch.__version__, torch.cuda.is_available(), pytorch_lightning.__version__)"', 'VERIFY')

RECOVERED PACKAGE RECORDS: 65
   torch = 1.13.1+cu117
   torchvision = 0.14.1+cu117
   pytorch-lightning = 1.6.5
   torchmetrics = 0.9.3
   transformers = 4.30.2
   sacred = 0.8.2
   pyarrow = 12.0.1
   timm = 0.6.13
   einops = 0.8.2
   ftfy = 6.3.1
   regex = 2026.1.15
   numpy = 1.23.5
   pillow = 9.5.0
   scikit-learn = not found
   scipy = not found
   pandas = 1.5.3
   tqdm = 4.70.0
   sentencepiece = not found
   tokenizers = 0.13.3
   jsonlines = 4.0.0
   opencv-python-headless = not found
SHELL TEST: SHELL_OK
STEP PYTHON39 -> exit 0
Installed Python 3.9.25 in 203ms
 + cpython-3.9.25-linux-x86_64-gnu (python3.9)
STEP VENV -> exit 0
Using CPython 3.9.25
Creating virtual environment at: dcpenv
Activate with: source dcpenv/bin/activate
INSTALL SPEC: torch==1.13.1+cu117 torchvision==0.14.1+cu117 pytorch-lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 sacred==0.8.2 pyarrow==12.0.1 timm==0.6.13 einops==0.8.2 ftfy==6.3.1 regex==2026.1.15 numpy==1.23.5 pillow==9.5.0 pandas==1

In [4]:
import os, subprocess

env = dict(os.environ, UV_PYTHON_INSTALL_DIR='/kaggle/working/uvpython')
P = '/kaggle/working/dcpenv/bin/python'

def run(c, tag):
    r = subprocess.run(c, shell=True, capture_output=True, text=True, env=env)
    print("STEP", tag, "-> exit", r.returncode)
    tail = (r.stdout[-700:] + r.stderr[-700:]).strip()
    if tail:
        print(tail)
    return r.returncode

run('uv pip install --python ' + P + ' --index-url https://download.pytorch.org/whl/cu117 '
    'torch==1.13.1+cu117 torchvision==0.14.1+cu117', 'TORCH')

run('uv pip install --python ' + P + ' '
    'pytorch-lightning==1.6.5 torchmetrics==0.9.3 transformers==4.30.2 sacred==0.8.2 '
    'pyarrow==12.0.1 timm==0.6.13 einops==0.8.2 ftfy==6.3.1 regex==2026.1.15 '
    'numpy==1.23.5 pillow==9.5.0 pandas==1.5.3 tqdm==4.70.0 tokenizers==0.13.3 '
    'jsonlines==4.0.0', 'REST')

run('chmod -R +x /kaggle/working/dcpenv/bin /kaggle/working/uvpython', 'CHMOD')

run(P + ' -c "import torch, pytorch_lightning, transformers, sacred, pyarrow, timm; '
    'print(\'VENV OK\', torch.__version__, torch.cuda.is_available(), pytorch_lightning.__version__)"', 'VERIFY')

STEP TORCH -> exit 0
Using Python 3.9.25 environment at: dcpenv
Resolved 10 packages in 10.75s
Prepared 10 packages in 25.99s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 10 packages in 4.47s
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + idna==3.4
 + numpy==2.0.2
 + pillow==11.3.0
 + requests==2.28.1
 + torch==1.13.1+cu117
 + torchvision==0.14.1+cu117
 + typing-extensions==4.16.0
 + urllib3==1.26.13
STEP REST -> exit 0
3
 + multidict==6.7.1
 + munch==2.5.0
 - numpy==2.0.2
 + numpy==1.23.5
 + packaging==26.3
 + pandas==1.5.3
 - pillow==11.3.0
 + pillow==9.5.0
 + propcache==0.4.1
 + protobuf==3.20.1
 + py-cpuinfo==9.0.0
 + pyarrow==12.0.1
 + pydeprecate==0.12.0
 + python-dateutil==2.9.0.post0
 + pytorch-lightning==1.6.5
 + pytz==2026.3.post1
 + pyyaml==6.0.3
 + regex==2026.1.15
 + sacred==0.8.

1

In [5]:
import os, subprocess

env = dict(os.environ, UV_PYTHON_INSTALL_DIR='/kaggle/working/uvpython')
P = '/kaggle/working/dcpenv/bin/python'

def run(c, tag):
    r = subprocess.run(c, shell=True, capture_output=True, text=True, env=env)
    print("STEP", tag, "-> exit", r.returncode)
    tail = (r.stdout[-700:] + r.stderr[-700:]).strip()
    if tail:
        print(tail)

run('uv pip install --python ' + P + ' "setuptools==68.2.2" "wheel"', 'SETUPTOOLS')

run(P + ' -c "import torch, pytorch_lightning, transformers, sacred, pyarrow, timm, ftfy; '
    'print(\'VENV OK\', torch.__version__, torch.cuda.is_available(), pytorch_lightning.__version__)"', 'VERIFY')

STEP SETUPTOOLS -> exit 0
Using Python 3.9.25 environment at: dcpenv
Resolved 3 packages in 117ms
Prepared 2 packages in 103ms
Uninstalled 1 package in 15ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 48ms
 - setuptools==82.0.1
 + setuptools==68.2.2
 + wheel==0.48.0
STEP VERIFY -> exit 0
VENV OK 1.13.1+cu117 False 1.6.5


In [1]:
import os, subprocess

P = '/kaggle/working/dcpenv/bin/python'
subprocess.run('chmod -R +x /kaggle/working/dcpenv/bin /kaggle/working/uvpython', shell=True)

r = subprocess.run(P + ' -c "import torch, pytorch_lightning; '
                   'print(\'READY\', torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())"',
                   shell=True, capture_output=True, text=True)
print("EXIT:", r.returncode)
print(r.stdout.strip())
print(r.stderr.strip()[-800:])

EXIT: 0




In [2]:
import subprocess, os, time

P = '/kaggle/working/dcpenv/bin/python'
subprocess.run('chmod -R +x /kaggle/working/dcpenv/bin /kaggle/working/uvpython', shell=True)

code = ("import torch, pytorch_lightning as pl\n"
        "print('PY ALIVE', flush=True)\n"
        "print('TORCH', torch.__version__, flush=True)\n"
        "print('PL', pl.__version__, flush=True)\n"
        "print('CUDA', torch.cuda.is_available(), torch.cuda.device_count(), flush=True)\n")
open('/kaggle/working/gpu_check.py', 'w').write(code)

os.system(P + ' -u /kaggle/working/gpu_check.py > /kaggle/working/gpu_check.log 2>&1')
time.sleep(2)
print("LOG:")
print(open('/kaggle/working/gpu_check.log', errors='ignore').read().strip() or "(empty)")
print()
os.system('nvidia-smi > /kaggle/working/smi.log 2>&1')
print("SMI:")
print(open('/kaggle/working/smi.log', errors='ignore').read().strip()[:900] or "(empty)")

LOG:
(empty)

SMI:
Sun Sep  6 23:23:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                       


In [4]:
import os, subprocess, stat

P = '/kaggle/working/dcpenv/bin/python'
print("P exists:", os.path.exists(P), "| islink:", os.path.islink(P))
print("realpath:", os.path.realpath(P))
print("real exists:", os.path.exists(os.path.realpath(P)))
print("executable bit:", os.access(P, os.X_OK), os.access(os.path.realpath(P), os.X_OK))

print()
print("dcpenv/bin:")
for f in sorted(os.listdir('/kaggle/working/dcpenv/bin'))[:20]:
    fp = '/kaggle/working/dcpenv/bin/' + f
    print("  ", f, "->", os.path.realpath(fp) if os.path.islink(fp) else "(file)")

print()
print("uvpython tree:")
for root, dirs, files in os.walk('/kaggle/working/uvpython'):
    d = root.replace('/kaggle/working/uvpython', '')
    if d.count('/') < 3:
        print("  ", d or '/', "|", len(dirs), "dirs", len(files), "files")

print()
rc = os.system(P + ' -c "print(1)" > /kaggle/working/t1.log 2>&1')
print("os.system rc:", rc, "| log:", repr(open('/kaggle/working/t1.log', errors='ignore').read()))

R = os.path.realpath(P)
rc2 = os.system(R + ' -c "print(2)" > /kaggle/working/t2.log 2>&1')
print("direct rc:", rc2, "| log:", repr(open('/kaggle/working/t2.log', errors='ignore').read()))

P exists: True | islink: False
realpath: /kaggle/working/dcpenv/bin/python
real exists: True
executable bit: True True

dcpenv/bin:
   activate -> (file)
   activate.bat -> (file)
   activate.csh -> (file)
   activate.fish -> (file)
   activate.nu -> (file)
   activate.ps1 -> (file)
   activate_this.py -> (file)
   convert-caffe2-to-onnx -> (file)
   convert-onnx-to-caffe2 -> (file)
   cpuinfo -> (file)
   deactivate.bat -> (file)
   f2py -> (file)
   f2py3 -> (file)
   f2py3.9 -> (file)
   ftfy -> (file)
   hf -> (file)
   huggingface-cli -> (file)
   markdown_py -> (file)
   normalizer -> (file)
   pydeprecate -> (file)

uvpython tree:
   / | 1 dirs 2 files
   /cpython-3.9.25-linux-x86_64-gnu | 4 dirs 1 files
   /cpython-3.9.25-linux-x86_64-gnu/lib | 7 dirs 4 files
   /cpython-3.9.25-linux-x86_64-gnu/share | 2 dirs 0 files
   /cpython-3.9.25-linux-x86_64-gnu/include | 1 dirs 0 files
   /cpython-3.9.25-linux-x86_64-gnu/bin | 0 dirs 8 files

os.system rc: 0 | log: ''
direct rc: 0 | log

In [5]:
import os

for p in ['/kaggle/working/dcpenv/bin/python',
          '/kaggle/working/dcpenv/bin/python3',
          '/kaggle/working/dcpenv/bin/python3.9',
          '/kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin/python3.9']:
    if os.path.exists(p):
        b = open(p, 'rb').read(120)
        print(p)
        print("   size:", os.path.getsize(p), "| head:", b[:60])
    else:
        print(p, "MISSING")

print()
print("uvpython bin files:")
ub = '/kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin'
for f in sorted(os.listdir(ub)):
    print("  ", f, os.path.getsize(os.path.join(ub, f)))

print()
os.system('echo SHELL_REDIRECT_OK > /kaggle/working/t3.log 2>&1')
print("shell redirect test:", repr(open('/kaggle/working/t3.log', errors='ignore').read()))

U = ub + '/python3.9'
os.system(U + ' -V > /kaggle/working/t4.log 2>&1')
print("base python -V:", repr(open('/kaggle/working/t4.log', errors='ignore').read()))

os.system('/kaggle/working/dcpenv/bin/python -V > /kaggle/working/t5.log 2>&1')
print("venv python -V:", repr(open('/kaggle/working/t5.log', errors='ignore').read()))

/kaggle/working/dcpenv/bin/python
   size: 0 | head: b''
/kaggle/working/dcpenv/bin/python3 MISSING
/kaggle/working/dcpenv/bin/python3.9 MISSING
/kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin/python3.9
   size: 18598712 | head: b'\x7fELF\x02\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x03\x00>\x00\x01\x00\x00\x00\x92-3\x00\x00\x00\x00\x00@\x00\x00\x00\x00\x00\x00\x00\xb8\xc3\x1b\x01\x00\x00\x00\x00\x00\x00\x00\x00@\x008\x00\x0c\x00@\x00'

uvpython bin files:
   2to3-3.9 157
   idle3.9 155
   pip 258
   pip3 258
   pip3.9 258
   pydoc3.9 140
   python3.9 18598712
   python3.9-config 3089

shell redirect test: 'SHELL_REDIRECT_OK\n'
base python -V: 'Python 3.9.25\n'
venv python -V: ''


In [1]:
import os

U = '/kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin/python3.9'
B = '/kaggle/working/dcpenv/bin'

for n in ['python', 'python3', 'python3.9']:
    t = os.path.join(B, n)
    if os.path.lexists(t):
        os.remove(t)
    os.symlink(U, t)
    print("LINKED", n, "->", os.path.realpath(t))

print("pyvenv.cfg:")
print(open('/kaggle/working/dcpenv/pyvenv.cfg').read().strip())

P = B + '/python'
os.system(P + ' -c "import sys, torch, pytorch_lightning as pl; '
          'print(\'READY\', sys.version.split()[0], torch.__version__, torch.cuda.is_available(), pl.__version__); '
          'print(\'SITE\', [x for x in sys.path if \'site-packages\' in x])" '
          '> /kaggle/working/ready.log 2>&1')
print()
print(open('/kaggle/working/ready.log', errors='ignore').read().strip() or "(empty)")

LINKED python -> /kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin/python3.9
LINKED python3 -> /kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin/python3.9
LINKED python3.9 -> /kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin/python3.9
pyvenv.cfg:
home = /kaggle/working/uvpython/cpython-3.9-linux-x86_64-gnu/bin
implementation = CPython
uv = 0.11.13
version_info = 3.9.25
include-system-site-packages = false

sh: 1: /kaggle/working/dcpenv/bin/python: Permission denied


In [2]:
import os, stat

U = '/kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin/python3.9'
UB = os.path.dirname(U)

for root, dirs, files in os.walk('/kaggle/working/uvpython'):
    for f in files:
        fp = os.path.join(root, f)
        if not os.path.islink(fp):
            try:
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
            except Exception:
                pass
print("uvpython exec bit:", os.access(U, os.X_OK))

for f in os.listdir('/kaggle/working/dcpenv/bin'):
    fp = '/kaggle/working/dcpenv/bin/' + f
    if not os.path.islink(fp):
        try:
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
        except Exception:
            pass

cfg = ("home = " + UB + "\n"
       "implementation = CPython\n"
       "uv = 0.11.13\n"
       "version_info = 3.9.25\n"
       "include-system-site-packages = false\n")
open('/kaggle/working/dcpenv/pyvenv.cfg', 'w').write(cfg)
print("pyvenv.cfg fixed, home =", UB)

P = '/kaggle/working/dcpenv/bin/python'
os.system(P + ' -c "import sys, torch, pytorch_lightning as pl; '
          'print(\'READY\', sys.version.split()[0], torch.__version__, torch.cuda.is_available(), pl.__version__); '
          'print(\'SITE\', [x for x in sys.path if \'site-packages\' in x])" '
          '> /kaggle/working/ready.log 2>&1')
print()
print(open('/kaggle/working/ready.log', errors='ignore').read().strip() or "(empty)")

uvpython exec bit: True
pyvenv.cfg fixed, home = /kaggle/working/uvpython/cpython-3.9.25-linux-x86_64-gnu/bin

READY 3.9.25 1.13.1+cu117 True 1.6.5
SITE ['/kaggle/working/dcpenv/lib/python3.9/site-packages']


In [3]:
import os, glob

print("RESULT DIRS:")
for d in sorted(os.listdir('/kaggle/working/dcp_results')):
    print("  ", d)

print()
h = glob.glob('/kaggle/working/dcp_results/*seed1*/version_*/hparams.yaml')
h = h or glob.glob('/kaggle/working/dcp_results/*/version_*/hparams.yaml')
print("HPARAMS FILE:", h[0] if h else "none")
if h:
    print(open(h[0]).read())

print()
print("HEAD OF mmimdb_s1.log:")
print(open('/kaggle/working/mmimdb_s1.log', errors='ignore').read()[:1500])

RESULT DIRS:
   base_fixed_seed0
   base_seed1_seed1
   gen2_seed0_seed0
   gen_seed0_seed0
   last_only_image_seed0
   last_only_text_seed0
   mmimdb_retr_seed0
   mmimdb_s1_seed1
   r2_image_0_seed0
   r2_image_1_seed0
   r2_text_0_seed0
   r2_text_1_seed0
   retr_check_seed0
   retr_image_0_seed0
   retr_image_1_seed0
   retr_text_0_seed0
   retr_text_1_seed0
   retr_train_seed0
   s1_only_image_seed0
   s1_only_text_seed0

HPARAMS FILE: /kaggle/working/dcp_results/base_seed1_seed1/version_0/hparams.yaml
config:
  batch_size: 256
  both_ratio: 0.5
  data_root: /kaggle/working/dl/arrow
  datasets:
  - Hatefull_Memes
  decay_power: 1
  draw_false_image: 0
  draw_false_text: 0
  end_lr: 0
  exp_name: base_seed1
  fast_dev_run: false
  finetune_first: false
  fix_model: true
  food101_class_num: 101
  get_recall_metric: false
  hatememes_class_num: 2
  image_only: false
  image_size: 224
  learning_rate: 0.01
  load_path: ''
  log_dir: /kaggle/working/dcp_results
  loss_names:
    food1

In [4]:
import os, subprocess, glob

P = '/kaggle/working/dcpenv/bin/python'
REPO = '/kaggle/working/Deep_Correlated_Prompting'
f = REPO + '/clip/modules/clip_missing_aware_prompt_module.py'
s = open(f).read()

print("PATCH CHECK")
for k in ['DCP_MGATE', 'gen_mgate', 'MODALITY GATE CREATED', 'DCP_DETECT', 'DCP_GEN', 'DCP_RETRIEVAL']:
    print("  ", k, "->", s.count(k))

r = subprocess.run(P + ' -m py_compile ' + f, shell=True, capture_output=True, text=True)
print("SYNTAX:", "OK" if r.returncode == 0 else r.stderr[-500:])

h = glob.glob('/kaggle/working/dcp_results/mmimdb_s1_seed1/version_*/hparams.yaml')
if h:
    hs = open(h[0]).read()
    print()
    print("MMIMDB CONFIG KEY LINES:")
    for line in hs.split(chr(10)):
        if any(w in line for w in ['exp_name', 'seed:', 'max_epoch', 'learning_rate', 'batch_size',
                                   'per_gpu', 'datasets', 'mmimdb', 'test_ratio', 'test_type', 'load_path']):
            print("  ", line.strip())

cmd = ('cd ' + REPO + ' && DCP_MGATE=1 ' + P + ' -u run.py with '
       'data_root=/kaggle/working/dl/arrow log_dir=/kaggle/working/dcp_results '
       'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 task_finetune_mmimdb_clip_prompt '
       'exp_name=mg_smoke seed=0 max_epoch=1 '
       '> /kaggle/working/mg_smoke.log 2>&1')
print()
print("RUNNING SMOKE TEST")
os.system(cmd)

log = open('/kaggle/working/mg_smoke.log', errors='ignore').read()
print("LOG LENGTH:", len(log))
import re
for pat in ['MODALITY GATE CREATED.*', 'MGATE mean g=.*', 'Trainable params.*',
            'val/the_metric.*', 'Error|Traceback|error']:
    m = re.findall(pat, log)
    print("FOUND" if m else "MISSING", pat, "|", (m[0][:120] if m else ""))
print()
print("LOG TAIL:")
print(log[-1200:])

PATCH CHECK
   DCP_MGATE -> 2
   gen_mgate -> 4
   MODALITY GATE CREATED -> 1
   DCP_DETECT -> 1
   DCP_GEN -> 3
   DCP_RETRIEVAL -> 1
SYNTAX: OK

MMIMDB CONFIG KEY LINES:
   batch_size: 256
   datasets:
   - mmimdb
   exp_name: mmimdb_s1
   learning_rate: 0.01
   load_path: ''
   mmimdb: 1
   max_epoch: 20
   missing_table_root: ./datasets/missing_tables/
   mmimdb_class_num: 23
   per_gpu_batchsize: 4
   seed: 1
   test_exp_name: null
   test_ratio: null
   test_type: null

RUNNING SMOKE TEST
LOG LENGTH: 570
MISSING MODALITY GATE CREATED.* | 
MISSING MGATE mean g=.* | 
MISSING Trainable params.* | 
MISSING val/the_metric.* | 
FOUND Error|Traceback|error | Error

LOG TAIL:
sacred.utils.NamedConfigNotFoundError: Named config not found: "task_finetune_mmimdb_clip_prompt". Available config values are: ('env_dandelin', 'task_mlm_itm', 'task_mlm_itm_randaug', 'task_mlm_itm_mpp', 'task_finetune_nlvr2', 'task_finetune_nlvr2_randaug', 'task_finetune_vqa', 'task_finetune_vqa_randaug', 'task_fi

In [5]:
import os, re

P = '/kaggle/working/dcpenv/bin/python'
REPO = '/kaggle/working/Deep_Correlated_Prompting'

cmd = ('cd ' + REPO + ' && DCP_MGATE=1 ' + P + ' -u run.py with '
       'data_root=/kaggle/working/dl/arrow log_dir=/kaggle/working/dcp_results '
       'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 task_finetune_mmimdb '
       'exp_name=mg_smoke seed=0 max_epoch=1 '
       '> /kaggle/working/mg_smoke.log 2>&1')
print("RUNNING SMOKE TEST")
os.system(cmd)

log = open('/kaggle/working/mg_smoke.log', errors='ignore').read()
print("LOG LENGTH:", len(log))
for pat in ['MODALITY GATE CREATED.*', 'MGATE mean g=.*', 'Trainable params.*',
            'val/the_metric.*', 'Traceback']:
    m = re.findall(pat, log)
    print("FOUND" if m else "MISSING", pat, "|", (m[0][:120] if m else ""))
print()
print("LOG TAIL:")
print(log[-1500:])

RUNNING SMOKE TEST
LOG LENGTH: 1621273
FOUND MODALITY GATE CREATED.* | MODALITY GATE CREATED | init g=0.5 = exact baseline
FOUND MGATE mean g=.* | MGATE mean g=0.5
FOUND Trainable params.* | Trainable params
FOUND val/the_metric.* | val/the_metric' reached 0.04139 (best 0.04139), saving model to '/kaggle/working/dcp_results/mg_smoke_seed0/version_0/ch
MISSING Traceback | 

LOG TAIL:
0:00, 17.92it/s]
Epoch 0: 100%|█████████▉| 7144/7148 [12:09<00:00,  9.79it/s, loss=0.269, v_num=0]

Validation DataLoader 0: 100%|██████████| 652/652 [00:39<00:00, 17.88it/s]

Epoch 0: 100%|█████████▉| 7145/7148 [12:10<00:00,  9.79it/s, loss=0.269, v_num=0]

                                                                          Epoch 0, global step 60: 'val/the_metric' was not in top 1

Epoch 0: 100%|██████████| 7148/7148 [12:12<00:00,  9.76it/s, loss=0.258, v_num=0]
INFO - CLIP - Completed after 0:12:22



In [6]:
import os, glob, re, time

P = '/kaggle/working/dcpenv/bin/python'
REPO = '/kaggle/working/Deep_Correlated_Prompting'
ROOT = '/kaggle/working/dcp_results'
t0 = time.time()

h = glob.glob(ROOT + '/s1_only_image_seed0/version_*/hparams.yaml') or glob.glob(ROOT + '/last_only_image_seed0/version_*/hparams.yaml')
if h:
    print("PREVIOUS EVAL CONFIG:")
    for line in open(h[0]).read().split(chr(10)):
        if any(w in line for w in ['test_only', 'test_ratio', 'test_type', 'load_path', 'exp_name', 'seed:']):
            print("  ", line.strip())
print()

train = ('cd ' + REPO + ' && DCP_MGATE=1 ' + P + ' -u run.py with '
         'data_root=/kaggle/working/dl/arrow log_dir=' + ROOT + ' '
         'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 task_finetune_mmimdb '
         'exp_name=mgate_seed0 seed=0 max_epoch=20 '
         '> /kaggle/working/mgate_seed0.log 2>&1')
print("TRAINING STARTED")
os.system(train)
print("TRAINING DONE in", round((time.time() - t0) / 60, 1), "minutes")

cks = [c for c in glob.glob(ROOT + '/mgate_seed0_seed0/version_*/checkpoints/*.ckpt') if 'last' not in c]
cks.sort(key=os.path.getmtime)
CK = cks[-1] if cks else ''
print("CHECKPOINT:", CK, round(os.path.getsize(CK) / 1e6, 1) if CK else 0, "MB")

if CK:
    for tt in ['both', 'image', 'text']:
        e = ('cd ' + REPO + ' && DCP_MGATE=1 ' + P + ' -u run.py with '
             'data_root=/kaggle/working/dl/arrow log_dir=' + ROOT + ' '
             'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 task_finetune_mmimdb '
             'exp_name=mg_eval_' + tt + ' seed=0 test_only=True test_ratio=0.7 '
             'test_type=' + tt + ' load_path=' + CK + ' '
             '> /kaggle/working/mg_eval_' + tt + '.log 2>&1')
        os.system(e)
        print("EVAL", tt, "DONE")

print()
print("RESULTS")
for tt in ['both', 'image', 'text']:
    p = '/kaggle/working/mg_eval_' + tt + '.log'
    if os.path.exists(p):
        s = open(p, errors='ignore').read()
        m = re.findall(r'F1_Macro[^\n]*|the_metric[^\n]*|DUMPED-TEST[^\n]*|Traceback', s)
        print(tt.upper(), "->", m[-4:] if m else "NO METRIC FOUND")

print()
print("TOTAL MINUTES:", round((time.time() - t0) / 60, 1))

PREVIOUS EVAL CONFIG:
   exp_name: s1_only_image
   load_path: /kaggle/working/dcp_results/base_seed1_seed1/version_0/checkpoints/epoch=8-step=306.ckpt
   seed: 0
   test_exp_name: null
   test_only: true
   test_ratio: 1.0
   test_type: image

TRAINING STARTED
TRAINING DONE in 250.1 minutes
CHECKPOINT: /kaggle/working/dcp_results/mgate_seed0_seed0/version_0/checkpoints/epoch=5-step=317.ckpt 673.5 MB
EVAL both DONE
EVAL image DONE
EVAL text DONE

RESULTS
BOTH -> ['DUMPED-TEST mg_eval_both', 'F1_Macro_epoch       0.5214624404907227', 'the_metric            0.5214624404907227']
IMAGE -> ['DUMPED-TEST mg_eval_image', 'F1_Macro_epoch       0.5378521680831909', 'the_metric            0.5378521680831909']
TEXT -> ['DUMPED-TEST mg_eval_text', 'F1_Macro_epoch       0.4964245855808258', 'the_metric            0.4964245855808258']

TOTAL MINUTES: 256.5


In [8]:
import os, glob, re

P = '/kaggle/working/dcpenv/bin/python'
REPO = '/kaggle/working/Deep_Correlated_Prompting'
ROOT = '/kaggle/working/dcp_results'
CK = '/kaggle/working/dcp_results/mgate_seed0_seed0/version_0/checkpoints/epoch=5-step=317.ckpt'

print("EXISTING DUMPS:")
for f in sorted(os.listdir('/kaggle/working/dumps')):
    print("  ", f, round(os.path.getsize('/kaggle/working/dumps/' + f) / 1e3, 1), "KB")

print()
for tt in ['both', 'image', 'text']:
    e = ('cd ' + REPO + ' && DCP_MGATE=1 DCP_EVAL_ON_DEV=1 ' + P + ' -u run.py with '
         'data_root=/kaggle/working/dl/arrow log_dir=' + ROOT + ' '
         'num_gpus=1 num_nodes=1 per_gpu_batchsize=4 task_finetune_mmimdb '
         'exp_name=mg_dev_' + tt + ' seed=0 test_only=True test_ratio=0.7 '
         'test_type=' + tt + ' load_path=' + CK + ' '
         '> /kaggle/working/mg_dev_' + tt + '.log 2>&1')
    os.system(e)
    s = open('/kaggle/working/mg_dev_' + tt + '.log', errors='ignore').read()
    m = re.findall(r'DUMPED-TEST[^\n]*|the_metric[^\n]*', s)
    print("DEV EVAL", tt, "->", m[-2:] if m else "FAILED")

print()
print("DUMPS AFTER DEV RUNS:")
for f in sorted(os.listdir('/kaggle/working/dumps')):
    if f.startswith('mg_'):
        print("  ", f, round(os.path.getsize('/kaggle/working/dumps/' + f) / 1e3, 1), "KB")

EXISTING DUMPS:
   devmmimdb_base_both.npz 480.4 KB
   devmmimdb_base_image.npz 480.4 KB
   devmmimdb_base_text.npz 480.4 KB
   devmmimdb_retr_both.npz 480.4 KB
   devmmimdb_retr_image.npz 480.4 KB
   devmmimdb_retr_text.npz 480.4 KB
   mg_eval_both.npz 1435.5 KB
   mg_eval_image.npz 1435.5 KB
   mg_eval_text.npz 1435.5 KB
   mmimdb_base_both.npz 1435.5 KB
   mmimdb_base_image.npz 1435.5 KB
   mmimdb_base_text.npz 1435.5 KB
   mmimdb_lie_image_1.npz 1435.5 KB
   mmimdb_lie_image_None.npz 1435.5 KB
   mmimdb_lie_text_2.npz 1435.5 KB
   mmimdb_lie_text_None.npz 1435.5 KB
   mmimdb_retr_both.npz 1435.5 KB
   mmimdb_retr_image.npz 1435.5 KB
   mmimdb_retr_text.npz 1435.5 KB
   s1_det4_both.npz 1435.5 KB
   s1_det4_image.npz 1435.5 KB
   s1_det4_text.npz 1435.5 KB
   s1_lie_image.npz 1435.5 KB
   s1_lie_text.npz 1435.5 KB
   s1_oracle_both.npz 1435.5 KB
   s1_oracle_image.npz 1435.5 KB
   s1_oracle_text.npz 1435.5 KB

DEV EVAL both -> ['DUMPED-TEST mg_dev_both', 'the_metric            0.528

In [1]:
import numpy as np
from sklearn.metrics import f1_score

D = '/kaggle/working/dumps/'

def load(n):
    z = np.load(D + n + '.npz')
    k = list(z.keys())
    lg = z[[x for x in k if 'log' in x.lower() or 'pred' in x.lower()][0]]
    lb = z[[x for x in k if 'lab' in x.lower() or 'target' in x.lower() or 'gt' in x.lower()][0]]
    return 1 / (1 + np.exp(-lg.astype(np.float64))), lb.astype(int), k

_, _, keys = load('mg_eval_both')
print("NPZ KEYS:", keys)
print()

def best_global(pd_, yd):
    best, bt = -1, 0.5
    for t in np.arange(0.05, 0.96, 0.01):
        f = f1_score(yd, (pd_ >= t).astype(int), average='macro', zero_division=0)
        if f > best:
            best, bt = f, t
    return bt

def best_perclass(pd_, yd):
    T = np.full(pd_.shape[1], 0.5)
    for c in range(pd_.shape[1]):
        best = -1
        for t in np.arange(0.05, 0.96, 0.01):
            f = f1_score(yd[:, c], (pd_[:, c] >= t).astype(int), zero_division=0)
            if f > best:
                best, T[c] = f, t
    return T

rows = []
for tag, devn, testn in [('BASE', 'devmmimdb_base_', 'mmimdb_base_'), ('GATE', 'mg_dev_', 'mg_eval_')]:
    for tt in ['image', 'text', 'both']:
        pdv, ydv, _ = load(devn + tt)
        pte, yte, _ = load(testn + tt)
        f05 = f1_score(yte, (pte >= 0.5).astype(int), average='macro', zero_division=0) * 100
        g = best_global(pdv, ydv)
        fg = f1_score(yte, (pte >= g).astype(int), average='macro', zero_division=0) * 100
        T = best_perclass(pdv, ydv)
        fp = f1_score(yte, (pte >= T).astype(int), average='macro', zero_division=0) * 100
        rows.append((tag, tt, f05, g, fg, fp))
        print(tag, tt, "| F1@0.5", round(f05, 2), "| thr", round(g, 2), "-> global", round(fg, 2), "| per-class", round(fp, 2))

print()
print("GATE MINUS BASE")
for tt in ['image', 'text', 'both']:
    b = [r for r in rows if r[0] == 'BASE' and r[1] == tt][0]
    g = [r for r in rows if r[0] == 'GATE' and r[1] == tt][0]
    print(" ", tt, "| raw", round(g[2] - b[2], 2), "| global-thr", round(g[4] - b[4], 2), "| per-class-thr", round(g[5] - b[5], 2))

NPZ KEYS: ['logits', 'labels']

BASE image | F1@0.5 53.73 | thr 0.32 -> global 56.09 | per-class 56.57
BASE text | F1@0.5 49.69 | thr 0.38 -> global 51.07 | per-class 51.79
BASE both | F1@0.5 51.44 | thr 0.31 -> global 53.61 | per-class 53.22
GATE image | F1@0.5 53.78 | thr 0.35 -> global 55.74 | per-class 55.97
GATE text | F1@0.5 49.64 | thr 0.24 -> global 50.35 | per-class 51.16
GATE both | F1@0.5 52.15 | thr 0.35 -> global 53.89 | per-class 53.32

GATE MINUS BASE
  image | raw 0.05 | global-thr -0.35 | per-class-thr -0.6
  text | raw -0.05 | global-thr -0.72 | per-class-thr -0.63
  both | raw 0.71 | global-thr 0.29 | per-class-thr 0.1
